# Processing Frog Spectrograms--Multiple Burst Types

Some frog species alternate between different kinds of bursts in one
recording, which `Processing_Frog_Spectrograms.ipynb` has no way to see since
it collapses every burst into one summary. This standalone variant adds two
things: burst-type clustering (group bursts by count/length/spacing, same
k-means + silhouette approach as the gap classification, generalized to more
than two groups) and a rescue pass for bursts too quiet to clear the global
threshold.

Produces one row per (clip, burst type), ten columns:

| Output Column | Description |
|---|---|
| `Element_Length` | Mean element (pulse) duration in seconds, for bursts of this type |
| `Inter-Element_Interval` | Mean within-burst gap in seconds, for bursts of this type |
| `Inter-Burst_Interval` | Mean between-burst gap in seconds for the whole clip; 0 for non-burst species |
| `Elements_Per_Burst` | Median elements per burst, for bursts of this type; 1 for non-burst species |
| `Min_Elements_Per_Burst` | Fewest elements found in any one burst of this type; 1 for non-burst species |
| `Max_Elements_Per_Burst` | Most elements found in any one burst of this type; 1 for non-burst species |
| `Burst_Type` | Letter (A, B, C...) for this burst type, in order of first appearance |
| `Burst_Pattern` | The clip's full burst-type sequence, e.g. `"ABCBA"` |
| `N_Bursts_This_Type` | How many bursts of this type occurred in the clip |
| `N_Burst_Types_Detected` | Total distinct burst types found (1 if no alternation) |

Otherwise identical to the original frog pipeline--no OCR, thin centerline
band for ink. This notebook also adds a handful of file-scoped fixes on top
(missed-element rescue, close-element splitting, and a few others).

**Run order:** `Frog_Clip_Log.ipynb` → this notebook

Stored as `frog_final_multi`, kept separate from `frog_final` so this
notebook never touches `Processing_Frog_Spectrograms.ipynb`'s results.

In [ ]:
import csv
import os
import re
import statistics
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.signal import find_peaks

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

# Cropped audio clips live here; generated oscillogram PNGs go in a separate folder.
BASE_DIR = Path.home() / 'Discrete_Signals'
AUDIO_DIR = BASE_DIR / 'Cropped_Frogs_Audios'
SPEC_DIR = BASE_DIR / 'Cropped_Frogs_Specs'
SPEC_DIR.mkdir(parents=True, exist_ok=True)

## Constants

In [ ]:
# Height fraction skipped around the zero-amplitude baseline when measuring
# ink--just enough to clear the always-dark line, not real pulse signal.
CENTER_EXCLUDE_FRACTION = 0.03

# Oscillogram DPI; fixed so pixel_time = clip_duration / image_width holds.
SAVE_DPI = 150

# Species with long call elements relative to the between-call pause, where
# segment_bursts' element-length-relative guards wrongly reject real burst
# structure.
LONG_ELEMENT_GAP_SPECIES = {"Adenomera_andreae", "Adenomera_saci", "Afrixalus_fulvovittatus"}

# Species confirmed to be isolated single calls separated by long silence;
# opts segment_bursts into its "split at every gap" fallback. Opt-in only
# (it over-splits on continuous background noise).
ISOLATED_CALL_SPECIES = {"Adenomera_engelsi", "Phasmahyla_guttata", "Ololygon_littoralis", "Adenomera_hylaedactyla", "Alytes_cisternasii", "Alytes_obstetricans"}

# Species dropped from the burst-type frequency feature entirely (beyond
# compute_burst_frequencies' own reliability gate). Amerana_draytonii's noise
# floor makes its per-burst pitch estimates unreliable.
FREQUENCY_FEATURE_EXCLUDED_SPECIES = {"Amerana_draytonii"}

# Files whose noise floor overlaps real call amplitude for the whole clip, so
# element counts are meaningless. Skipped rather than reported with bad numbers
# (seven general-fix attempts failed.).
NOISE_FLOOR_EXCLUDED_FILES = {
    "Amerana_draytonii_32", "Amerana_draytonii_33", "Amerana_draytonii_34", "Amerana_draytonii_317", "Amerana_draytonii_318", "Amerana_draytonii_319",
    "Amerana_draytonii_320", "Amerana_draytonii_323", "Amerana_draytonii_324", "Amerana_draytonii_325", "Amerana_draytonii_327", "Amerana_draytonii_329",
    "Amerana_draytonii_330", "Amerana_draytonii_331", "Amerana_draytonii_332", "Amerana_draytonii_334", "Amerana_draytonii_335", "Amerana_draytonii_336",
    "Amerana_draytonii_337", "Amerana_draytonii_338", "Amerana_draytonii_341", "Amerana_draytonii_342", "Amerana_draytonii_343", "Amerana_draytonii_344",
    "Amerana_draytonii_345", "Amerana_draytonii_346", "Amerana_draytonii_347", "Amerana_draytonii_348", "Amerana_draytonii_349", "Amerana_draytonii_350",
    "Amerana_draytonii_351", "Amerana_draytonii_352", "Amerana_draytonii_354", "Amerana_draytonii_355", "Amerana_draytonii_356", "Amerana_draytonii_357",
    "Amerana_draytonii_358", "Amerana_draytonii_359", "Amerana_draytonii_360", "Amerana_draytonii_361", "Amerana_draytonii_362", "Amerana_draytonii_363",
    "Amerana_draytonii_378", "Amerana_draytonii_379", "Amerana_draytonii_381", "Amerana_draytonii_382", "Amerana_draytonii_383", "Amerana_draytonii_384",
    "Amerana_draytonii_386", "Amerana_draytonii_387", "Amerana_draytonii_391", "Amerana_draytonii_394", "Amerana_draytonii_397", "Amerana_draytonii_403",
    "Amerana_draytonii_404", "Amerana_draytonii_405", "Amerana_draytonii_407", "Amerana_draytonii_408", "Amerana_draytonii_414", "Amerana_draytonii_415",
    "Amerana_draytonii_417", "Amerana_draytonii_419", "Amerana_draytonii_423", "Amerana_draytonii_429", "Amerana_draytonii_430", "Amerana_draytonii_431",
    "Amerana_draytonii_432", "Amerana_draytonii_433", "Amerana_draytonii_434", "Amerana_draytonii_435", "Amerana_draytonii_436", "Amerana_draytonii_437",
    "Amerana_draytonii_439", "Amerana_draytonii_442", "Amerana_draytonii_443", "Amerana_draytonii_444", "Amerana_draytonii_445", "Amerana_draytonii_446",
    "Amerana_draytonii_447", "Amerana_draytonii_448", "Amerana_draytonii_449", "Amerana_draytonii_450", "Amerana_draytonii_451", "Amerana_draytonii_452",
    "Amerana_draytonii_453", "Amerana_draytonii_454", "Amerana_draytonii_455", "Amerana_draytonii_459", "Amerana_draytonii_460", "Amerana_draytonii_461",
    "Amerana_draytonii_462", "Amerana_draytonii_463", "Amerana_draytonii_466", "Amerana_draytonii_467", "Amerana_draytonii_468", "Amerana_draytonii_470",
    "Amerana_draytonii_471", "Amerana_draytonii_472", "Amerana_draytonii_473", "Amerana_draytonii_476", "Amerana_draytonii_477", "Amerana_draytonii_478",
    "Amerana_draytonii_479", "Amerana_draytonii_480", "Amerana_draytonii_481", "Amerana_draytonii_482", "Amerana_draytonii_484", "Amerana_draytonii_488",
    "Amerana_draytonii_489", "Amerana_draytonii_490", "Amerana_draytonii_491", "Amerana_draytonii_493", "Amerana_draytonii_494", "Amerana_draytonii_495",
    "Amerana_draytonii_496", "Amerana_draytonii_498", "Amerana_draytonii_499", "Amerana_draytonii_500", "Amerana_draytonii_501", "Amerana_draytonii_502",
    "Amerana_draytonii_503", "Amerana_draytonii_504", "Amerana_draytonii_505", "Amerana_draytonii_506", "Amerana_draytonii_507", "Amerana_draytonii_508",
    "Amerana_draytonii_509", "Amerana_draytonii_510", "Amerana_draytonii_511", "Amerana_draytonii_512", "Amerana_draytonii_514", "Amerana_draytonii_515",
    "Amerana_draytonii_516", "Amerana_draytonii_517", "Amerana_draytonii_518", "Amerana_draytonii_519", "Amerana_draytonii_520", "Amerana_draytonii_522",
    "Amerana_draytonii_523", "Amerana_draytonii_524", "Amerana_draytonii_526", "Amerana_draytonii_528", "Amerana_draytonii_529", "Amerana_draytonii_531",
    "Amerana_draytonii_532", "Amerana_draytonii_533", "Amerana_draytonii_534", "Amerana_draytonii_535", "Amerana_draytonii_537", "Amerana_draytonii_538",
    "Amerana_draytonii_539", "Amerana_draytonii_540", "Amerana_draytonii_541", "Amerana_draytonii_542", "Amerana_draytonii_543", "Amerana_draytonii_544",
    "Amerana_draytonii_549", "Amerana_draytonii_550", "Amerana_draytonii_551", "Amerana_draytonii_552", "Amerana_draytonii_553", "Amerana_draytonii_554",
    "Amerana_draytonii_555", "Amerana_draytonii_556", "Amerana_draytonii_557", "Amerana_draytonii_558", "Amerana_draytonii_560", "Amerana_draytonii_561",
    "Amerana_draytonii_562", "Amerana_draytonii_563", "Amerana_draytonii_565", "Amerana_draytonii_566", "Amerana_draytonii_567", "Amerana_draytonii_568",
    "Amerana_draytonii_578", "Amerana_draytonii_579", "Amerana_draytonii_580", "Amerana_draytonii_581", "Amerana_draytonii_582", "Amerana_draytonii_616",
    "Amerana_draytonii_617", "Amerana_draytonii_621", "Amerana_draytonii_623",
}

# Files carrying a hand-derived correction the ink-based detection can't
# reproduce. No automated mechanism, including the general fallback below,
# should touch these (they've been lost to CSV regenerations before). See
PROTECTED_HAND_FIXED_FILES = {
    "Alytes_dickhilleni_4", "Amerana_draytonii_156", "Amerana_draytonii_157",
    "Amerana_draytonii_16", "Amerana_draytonii_165", "Amerana_draytonii_18",
    "Amerana_draytonii_229", "Amerana_draytonii_233", "Amerana_draytonii_234",
    "Amerana_draytonii_235", "Amerana_draytonii_237", "Amerana_draytonii_238",
    "Amerana_draytonii_240", "Amerana_draytonii_241", "Amerana_draytonii_246",
    "Amerana_draytonii_248", "Amerana_draytonii_250", "Amerana_draytonii_252",
    "Amerana_draytonii_253", "Amerana_draytonii_254", "Amerana_draytonii_256",
    "Amerana_draytonii_259", "Amerana_draytonii_261", "Amerana_draytonii_262",
    "Amerana_draytonii_266", "Amerana_draytonii_267", "Amerana_draytonii_269",
    "Amerana_draytonii_272", "Amerana_draytonii_273", "Amerana_draytonii_274",
    "Amerana_draytonii_275", "Amerana_draytonii_276", "Amerana_draytonii_277",
    "Amerana_draytonii_278", "Amerana_draytonii_279", "Amerana_draytonii_280",
    "Amerana_draytonii_281", "Amerana_draytonii_282", "Amerana_draytonii_283",
    "Amerana_draytonii_284", "Amerana_draytonii_285", "Amerana_draytonii_286",
    "Amerana_draytonii_287", "Amerana_draytonii_288", "Amerana_draytonii_289",
    "Amerana_draytonii_291", "Amerana_draytonii_292", "Amerana_draytonii_293",
    "Amerana_draytonii_294", "Amerana_draytonii_295", "Amerana_draytonii_296",
    "Amerana_draytonii_297", "Amerana_draytonii_298", "Amerana_draytonii_299",
    "Amerana_draytonii_3", "Amerana_draytonii_30", "Amerana_draytonii_300",
    "Amerana_draytonii_301", "Amerana_draytonii_302", "Amerana_draytonii_303",
    "Amerana_draytonii_304", "Amerana_draytonii_305", "Amerana_draytonii_306",
    "Amerana_draytonii_307", "Amerana_draytonii_308", "Amerana_draytonii_31",
    "Amerana_draytonii_310", "Amerana_draytonii_315", "Amerana_draytonii_316",
}


## Load Clip Log

Reads `frog_clip_log.csv` directly rather than via `%store`. Keeps only the
`ok` rows and matches each one to its wav file on disk.

In [ ]:
clips_ready = pd.read_csv(AUDIO_DIR / 'frog_clip_log.csv')

for col in ('audio_num', 'source_dur_s', 'start_time_s', 'end_time_s', 'clip_duration_s'):
    clips_ready[col] = pd.to_numeric(clips_ready[col], errors='coerce')

clips_ready = clips_ready[clips_ready['status'] == 'ok'].copy()

# Match each row to its actual wav file (flat in AUDIO_DIR, not the recorded
# container path in the CSV, which no longer resolves on this machine).
clips_ready['wav_name'] = clips_ready.apply(
    lambda r: f"{r['genus']}_{r['species']}_{int(r['audio_num'])}.wav", axis=1
)
clips_ready['clipped_file'] = clips_ready['wav_name'].apply(lambda n: str(AUDIO_DIR / n))
clips_ready = clips_ready[clips_ready['clipped_file'].apply(lambda p: Path(p).exists())]
clips_ready = clips_ready.reset_index(drop=True)

print(f"{len(clips_ready)} clips loaded")
clips_ready.head()

## Helper Functions

Same as the cricket and katydid pipelines.

In [ ]:
def gaussian_filter1d(signal, sigma):
    """Apply a 1-D Gaussian smoothing kernel to a 1-D array."""
    kernel_radius    = int(4 * sigma + 0.5)
    kernel_positions = np.arange(-kernel_radius, kernel_radius + 1, dtype=float)
    kernel           = np.exp(-0.5 * (kernel_positions / sigma) ** 2)
    kernel          /= kernel.sum()
    return np.convolve(signal, kernel, mode="same")


def silhouette(values, cluster_labels):
    """Mean silhouette score for a 1-D binary clustering."""
    total_score = 0.0
    for i, value in enumerate(values):
        same_cluster  = values[cluster_labels == cluster_labels[i]]
        other_cluster = values[cluster_labels != cluster_labels[i]]
        within_dist   = np.mean(np.abs(same_cluster  - value)) if len(same_cluster)  > 1 else 0.0
        between_dist  = np.mean(np.abs(other_cluster - value)) if len(other_cluster) > 0 else 0.0
        denom         = max(within_dist, between_dist)
        total_score  += (between_dist - within_dist) / denom if denom > 0 else 0.0
    return total_score / len(values)


def kmeans2(values):
    """Best 2-way split of a 1-D array by trying every split point and keeping
    the highest-silhouette one. Returns (cluster_labels, cluster_centers);
    label 0 = short, 1 = long."""
    sorted_values    = np.sort(values)
    best_silhouette  = -2.0
    best_split_index = 1

    for i in range(1, len(sorted_values)):
        if i > 1 and sorted_values[i] == sorted_values[i - 1]:
            continue
        split_threshold  = (sorted_values[i - 1] + sorted_values[i]) / 2
        cluster_labels   = (values > split_threshold).astype(int)
        if len(set(cluster_labels)) < 2:
            continue
        split_silhouette = silhouette(values, cluster_labels)
        if split_silhouette > best_silhouette:
            best_silhouette  = split_silhouette
            best_split_index = i

    split_threshold = (sorted_values[best_split_index - 1] + sorted_values[best_split_index]) / 2
    cluster_labels  = (values > split_threshold).astype(int)
    cluster_centers = np.array([
        values[cluster_labels == 0].mean() if (cluster_labels == 0).any() else sorted_values[0],
        values[cluster_labels == 1].mean() if (cluster_labels == 1).any() else sorted_values[-1],
    ])
    return cluster_labels, cluster_centers


def choose_k(gap_durations):
    """
    Return 1 if gaps form one cluster, 2 if they form two distinct clusters.
    Requires silhouette > 0.3 before declaring burst structure.
    """
    gap_durations = np.asarray(gap_durations, dtype=float)
    if len(gap_durations) < 3 or len(np.unique(gap_durations)) < 2:
        return 1
    cluster_labels, _ = kmeans2(gap_durations)
    if len(set(cluster_labels)) < 2:
        return 1
    return 2 if silhouette(gap_durations, cluster_labels) > 0.3 else 1


def rle(signal_list):
    """Run-length encode a 1-D sequence."""
    run_list      = []
    current_value = signal_list[0]
    run_length    = 1
    for value in signal_list[1:]:
        if value == current_value:
            run_length += 1
        else:
            run_list.append((current_value, run_length))
            current_value = value
            run_length    = 1
    run_list.append((current_value, run_length))
    return run_list

## Signal Analysis

A few things tuned for frogs specifically: gentle smoothing (oscillograms
have pixel-level noise), no gap-filling (frog inter-element gaps are real
signal, not noise), and a slightly looser active-fraction ceiling since
silence still dominates most frog calls. Ink is measured around a thin band
near the detected centerline, found per-image rather than assumed to be
centered. Thresholding falls back through a standard ladder, then an extended
one for high-baseline recordings, then a continuous-trill catch-all--see
`detect_signal_list_adaptive`'s docstring.

In [ ]:
def clean_signal_runs(signal_list, pixel_time, fill_gap_size=0):
    """
    Remove on-runs shorter than 3 ms (spurious noise detections).

    fill_gap_size > 0  Fill off-gaps of <= that many pixels surrounded by
                       on-runs on both sides (not used for frogs by default).
    """
    signal_list   = np.asarray(signal_list, dtype=np.uint8)
    min_on_pixels = max(1, round(0.003 / pixel_time))

    cleaned = []
    for value, run_length in rle(signal_list):
        if value == 1 and run_length < min_on_pixels:
            cleaned.extend([0] * run_length)
        else:
            cleaned.extend([int(value)] * run_length)
    cleaned = np.array(cleaned, dtype=np.uint8)

    if fill_gap_size > 0 and len(cleaned) > 2:
        result_list = []
        run_list    = rle(cleaned)
        for i, (value, run_length) in enumerate(run_list):
            surrounded_by_signal = (
                i > 0 and i < len(run_list) - 1
                and run_list[i - 1][0] == 1 and run_list[i + 1][0] == 1
            )
            if value == 0 and surrounded_by_signal and run_length <= fill_gap_size:
                result_list.extend([1] * run_length)
            else:
                result_list.extend([int(value)] * run_length)
        cleaned = np.array(result_list, dtype=np.uint8)

    return cleaned


def find_centerline_row(spec_array):
    """Row of the always-dark zero-amplitude baseline, found by darkness rather
    than assumed to be the image's middle (a tight bbox can crop asymmetrically)."""
    row_dark_fraction = (spec_array < 200).mean(axis=1)
    return int(np.argmax(row_dark_fraction))


def extract_outer_band_ink(spec_array):
    """2-D ink array with only a thin band around the detected centerline
    excluded (~1-2% of height), so quiet near-baseline elements still register.
    An earlier version excluded the inner 40% and silently dropped them."""
    img_height, img_width = spec_array.shape
    center_row    = find_centerline_row(spec_array)
    half_exclude  = max(1, int(img_height * CENTER_EXCLUDE_FRACTION / 2))
    rows          = np.arange(img_height)
    keep_mask     = np.abs(rows - center_row) > half_exclude
    band          = spec_array[keep_mask, :].astype(float)

    # Ink = darkness relative to background (white background -> near-zero ink in gaps)
    background_level = np.percentile(band, 95)
    ink              = np.clip(background_level - band, 0, None)
    ink_peak         = np.percentile(ink, 99)
    if ink_peak > 0:
        ink /= ink_peak

    return ink


def detect_signal_list_adaptive(ink_array, pixel_time):
    """Threshold the ink array into a binary on/off signal via adaptive
    hysteresis, best threshold scored by (high - noise penalty). Walks a
    standard ladder, then an extended high-baseline one (kept separate so its
    larger values don't win on magnitude), then falls back to a single
    continuous on-run. Frog tuning: sigma=0.45, active_fraction < 0.90,
    fill_gap_size=0. Returns (signal_list, normalized_signal)."""
    col_pct85           = np.percentile(ink_array, 85, axis=0)
    col_pct95           = np.percentile(ink_array, 95, axis=0)
    col_pct99           = np.percentile(ink_array, 99, axis=0)
    col_signal_strength = gaussian_filter1d(
        0.20 * col_pct85 + 0.35 * col_pct95 + 0.45 * col_pct99,
        sigma=0.45
    )

    signal_floor      = np.percentile(col_signal_strength,  5)
    signal_peak       = np.percentile(col_signal_strength, 99.5)
    if signal_peak <= signal_floor:
        if col_signal_strength.max() >= 0.5:
            # Saturated, not blank: a continuous near-clipping trill collapses
            # both percentiles to the same value from the top. Treat it like the
            # >=90%-active fallback below rather than erroring (this used to drop
            # the file as "blank").
            signal_list = clean_signal_runs(
                np.ones(len(col_signal_strength), dtype=np.uint8), pixel_time, fill_gap_size=0)
            return signal_list, np.ones(len(col_signal_strength))
        raise ValueError("No contrast in ink array--image may be blank")
    normalized_signal = np.clip(
        (col_signal_strength - signal_floor) / (signal_peak - signal_floor), 0, 1
    )

    min_on_pixels = max(1, round(0.003 / pixel_time))

    def try_ladder(ladder):
        candidates = []
        for high_threshold, low_threshold in ladder:
            high_mask = normalized_signal >= high_threshold
            low_mask  = normalized_signal >= low_threshold

            signal_list = np.zeros_like(low_mask, dtype=np.uint8)
            column_pos  = 0
            for value, run_length in rle(low_mask.astype(np.uint8)):
                run_start, run_end = column_pos, column_pos + run_length
                if value == 1 and run_length >= min_on_pixels and np.any(high_mask[run_start:run_end]):
                    signal_list[run_start:run_end] = 1
                column_pos = run_end

            signal_list     = clean_signal_runs(signal_list, pixel_time, fill_gap_size=0)
            active_fraction = float(signal_list.mean())
            if active_fraction <= 0 or active_fraction >= 0.90:
                continue

            on_run_lengths = [run_length for value, run_length in rle(signal_list) if value == 1]
            if not on_run_lengths:
                continue

            median_on_length  = float(np.median(on_run_lengths))
            tiny_run_fraction = sum(l <= 1 for l in on_run_lengths) / len(on_run_lengths)
            score = high_threshold - 0.15 * tiny_run_fraction - 0.002 * median_on_length
            candidates.append((score, signal_list))
        return candidates

    standard_ladder = [(t, max(0.06, t * 0.45)) for t in
                        [0.78, 0.70, 0.62, 0.54, 0.46, 0.38, 0.30, 0.22, 0.16, 0.10]]
    candidate_thresholds = try_ladder(standard_ladder)

    if not candidate_thresholds:
        extended_ladder = [(t, max(0.06, t - 0.12)) for t in [0.98, 0.94, 0.90, 0.86, 0.82]]
        candidate_thresholds = try_ladder(extended_ladder)

    if not candidate_thresholds:
        # Every threshold in both ladders produced active_fraction >= 0.90:
        # this is a genuine continuous trill, not a detection failure. Fall
        # back to the lowest threshold without the active-fraction ceiling so
        # the caller sees one long on-run rather than an error.
        low_threshold = 0.10
        low_mask      = normalized_signal >= low_threshold
        signal_list   = clean_signal_runs(low_mask.astype(np.uint8), pixel_time, fill_gap_size=0)
        if signal_list.mean() <= 0:
            raise ValueError("No valid signal detected at any threshold")
        return signal_list, normalized_signal

    candidate_thresholds.sort(key=lambda t: t[0], reverse=True)
    return candidate_thresholds[0][1], normalized_signal

## Low-Volume Burst Rescue

New in this notebook, same rationale as the katydid version: ink and signal
thresholds are normalized against the whole image, so a burst much quieter
than another one in the same clip can get compressed toward zero and never
cross the threshold. `rescue_quiet_bursts` re-checks whatever the global pass
missed, re-crops any region with real (if faint) ink, and re-runs detection
on that crop alone--normalized against its own loudness instead of
competing with the rest of the recording.

In [ ]:
def group_consecutive(indices, max_gap_px):
    """
    Group a sorted 1-D array of column indices into contiguous regions, merging
    regions separated by at most max_gap_px columns.

    Returns a list of (start, end) tuples, inclusive on both ends.
    """
    if not len(indices):
        return []
    regions = []
    start = prev = indices[0]
    for idx in indices[1:]:
        if idx - prev <= max_gap_px:
            prev = idx
        else:
            regions.append((start, prev))
            start = prev = idx
    regions.append((start, prev))
    return regions


def rescue_quiet_bursts(spec_array, ink, raw_signal_list, pixel_time,
                         presence_threshold=0.03, max_gap_px=8,
                         min_region_px=4, pad_px=3):
    """Recover bursts real but too quiet (vs. a louder burst in the same
    recording) to clear the global threshold--ink is normalized from
    whole-image percentiles. Finds columns with real ink the global pass
    missed, then re-crops and re-thresholds each region locally. See the
    markdown above. Returns raw_signal_list OR'd with anything rescued."""
    combined = raw_signal_list.copy()
    presence_cols = np.where(ink.max(axis=0) > presence_threshold)[0]
    missed_cols = presence_cols[combined[presence_cols] == 0]
    if not len(missed_cols):
        return combined

    for region_start, region_end in group_consecutive(missed_cols, max_gap_px):
        if region_end - region_start + 1 < min_region_px:
            continue

        pad_start = max(0, region_start - pad_px)
        pad_end   = min(spec_array.shape[1], region_end + pad_px + 1)

        if combined[pad_start:pad_end].any():
            continue

        crop = spec_array[:, pad_start:pad_end]
        try:
            local_ink = extract_outer_band_ink(crop)
            local_signal, _ = detect_signal_list_adaptive(local_ink, pixel_time)
        except ValueError:
            continue

        combined[pad_start:pad_end] = np.maximum(combined[pad_start:pad_end], local_signal)

    return combined

## Interval Classification & Burst-Type Clustering

Finding burst boundaries works exactly like the single-burst notebook--same
gap clustering, refactored (`segment_bursts`) to return each burst's own
stats instead of one recording-wide summary.

What's new: a species can alternate between different kinds of bursts in one
recording. `classify_burst_types` clusters the bursts `segment_bursts` finds
using the same k-means + silhouette technique, generalized to handle more
than one feature and more than two groups.

In [ ]:
def silhouette_nd(features, cluster_labels):
    """
    Mean silhouette score for a multi-dimensional clustering -- the same idea as
    silhouette() above, generalized from 1-D absolute difference to Euclidean
    distance so it works on multi-feature burst vectors instead of single gap
    durations.
    """
    total_score = 0.0
    for i, point in enumerate(features):
        same_cluster  = features[cluster_labels == cluster_labels[i]]
        other_cluster = features[cluster_labels != cluster_labels[i]]
        within_dist  = (
            np.mean(np.linalg.norm(same_cluster - point, axis=1))
            if len(same_cluster) > 1 else 0.0
        )
        between_dist = (
            np.mean(np.linalg.norm(other_cluster - point, axis=1))
            if len(other_cluster) > 0 else 0.0
        )
        denom = max(within_dist, between_dist)
        total_score += (between_dist - within_dist) / denom if denom > 0 else 0.0
    return total_score / len(features)


def kmeans_nd(features, k, n_init=8, max_iter=100, random_seed=0):
    """Random-restart Lloyd's k-means for multi-dimensional points (kmeans2's
    exhaustive 1-D search doesn't generalize past one feature / two clusters).
    Returns one cluster label per row of `features`."""
    rng = np.random.default_rng(random_seed)
    n = len(features)
    best_labels, best_inertia = None, np.inf

    for _ in range(n_init):
        centroid_idx = rng.choice(n, size=k, replace=False)
        centroids = features[centroid_idx].copy()

        for _ in range(max_iter):
            distances = np.linalg.norm(features[:, None, :] - centroids[None, :, :], axis=2)
            labels = distances.argmin(axis=1)

            new_centroids = centroids.copy()
            for cluster_id in range(k):
                members = features[labels == cluster_id]
                if len(members):
                    new_centroids[cluster_id] = members.mean(axis=0)

            if np.allclose(new_centroids, centroids):
                centroids = new_centroids
                break
            centroids = new_centroids

        distances = np.linalg.norm(features[:, None, :] - centroids[None, :, :], axis=2)
        labels = distances.argmin(axis=1)
        inertia = float(np.sum((features - centroids[labels]) ** 2))

        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = labels

    return best_labels


def choose_k_burst_types(features, max_k=4, min_cluster_size=2, n_init=8):
    """Multi-dimensional, multi-k choose_k: try k = 2..max_k, keep the best
    silhouette above 0.3, else one cluster. min_cluster_size rejects singleton
    "types"--a burst type should mean "this kind of burst happened more than
    once". Returns one cluster label per row of `features`."""
    n = len(features)
    max_k = min(max_k, n // min_cluster_size)
    if n < 3 or max_k < 2:
        return np.zeros(n, dtype=int)

    best_labels, best_score = np.zeros(n, dtype=int), -2.0
    for k in range(2, max_k + 1):
        labels = kmeans_nd(features, k, n_init=n_init)
        cluster_sizes = np.bincount(labels, minlength=k)
        if (cluster_sizes < min_cluster_size).any():
            continue
        score = silhouette_nd(features, labels)
        if score > best_score:
            best_labels, best_score = labels, score

    if best_score <= 0.3:
        return np.zeros(n, dtype=int)
    return best_labels


In [ ]:
def segment_bursts(time_bucket_list, leading_silence, trailing_silence,
                    element_length, pixel_time, relaxed_ratio=False, isolated_burst_ok=False,
                    force_element_level_split=False):
    """
    Split time_bucket_list into individual bursts. Same gap classification
    (kmeans2/choose_k on "short" vs. "long" gaps) and burst-boundary guards
    as the original classify_intervals in Processing_Frog_Spectrograms.ipynb,
    but returns each burst's own stats instead of one recording-wide
    summary -- that's what classify_burst_types clusters on below.

    Returns (inter_element_interval, inter_burst_interval, bursts), where
    bursts is a list of dicts, one per burst (or a single dict spanning the
    whole recording if no burst structure was found):
        {'element_count': int, 'mean_element_length': float,
         'mean_inter_element_interval': float, 'start_time': float, 'end_time': float}
    """
    def element_level_bursts():
        on_durs = [d for s, d in time_bucket_list if s == "On"]
        off_durs = [d for s, d in time_bucket_list if s == "Off"]
        mean_gap = float(np.mean(off_durs)) if off_durs else 0.0
        bursts = []
        elapsed = 0.0
        for state, dur in time_bucket_list:
            if state == "On":
                bursts.append({
                    'element_count': 1,
                    'mean_element_length': dur,
                    'mean_inter_element_interval': mean_gap,
                    'start_time': elapsed,
                    'end_time': elapsed + dur,
                })
            elapsed += dur
        return mean_gap, 0.0, bursts

    if force_element_level_split:
        return element_level_bursts()

    def whole_recording_as_one_burst():
        on_durs  = [d for s, d in time_bucket_list if s == "On"]
        off_durs = [d for s, d in time_bucket_list if s == "Off"]
        total_time = leading_silence + sum(d for _, d in time_bucket_list) + trailing_silence
        return [{
            'element_count':               len(on_durs),
            'mean_element_length':         float(np.mean(on_durs)) if on_durs else 0.0,
            'mean_inter_element_interval': float(np.mean(off_durs)) if off_durs else 0.0,
            'start_time':                  0.0,
            'end_time':                    total_time,
        }]

    off_durations = np.array(
        [duration for state, duration in time_bucket_list if state == "Off"],
        dtype=float,
    )

    if len(off_durations) == 0:
        edge_gaps = [x for x in [leading_silence, trailing_silence] if x > 0]
        return 0.0, float(np.mean(edge_gaps)) if edge_gaps else 0.0, whole_recording_as_one_burst()

    def split_at_every_gap():
        """
        Treat every off-run as a burst boundary, so each on-run becomes its
        own single-element burst. Used when every gap is already huge
        relative to element length -- a series of isolated, widely-spaced
        single calls (e.g. Adenomera_engelsi_1), not one continuous burst
        the old fallback would otherwise misrepresent as a single grouped
        burst. Only fires at an extreme gap/element ratio, so continuous
        trills never reach it.
        """
        result_bursts, current_on_durs, current_start = [], [], 0.0
        elapsed = 0.0
        for state, duration in time_bucket_list:
            if state == "On":
                if not current_on_durs:
                    current_start = elapsed
                current_on_durs.append(duration)
            else:
                if current_on_durs:
                    result_bursts.append({
                        'element_count':               len(current_on_durs),
                        'mean_element_length':          float(np.mean(current_on_durs)),
                        'mean_inter_element_interval':  0.0,
                        'start_time': current_start,
                        'end_time':   elapsed,
                    })
                current_on_durs = []
            elapsed += duration
        if current_on_durs:
            result_bursts.append({
                'element_count':               len(current_on_durs),
                'mean_element_length':          float(np.mean(current_on_durs)),
                'mean_inter_element_interval':  0.0,
                'start_time': current_start,
                'end_time':   elapsed,
            })
        return result_bursts

    # ISOLATED_CALL_SPECIES' split-every-gap fallback runs only as a last
    # resort, after normal clustering--running it first wrongly overrode real
    # multi-element bursts.
    def isolated_fallback_if_applicable():
        # median, not mean: a couple of tiny edge-padding gaps left over from
        # trimming can drag the mean below the ratio threshold even when the
        # real gaps clearly qualify. median is robust to those outliers.
        gap_stat = float(np.median(off_durations))
        if (
            isolated_burst_ok
            and len(off_durations) >= 2
            and gap_stat >= 8 * element_length
        ):
            isolated_bursts = split_at_every_gap()
            if isolated_bursts:
                return 0.0, gap_stat, isolated_bursts
        return float(np.mean(off_durations)), 0.0, whole_recording_as_one_burst()

    if len(off_durations) < 4 or len(np.unique(np.round(off_durations, 6))) < 2:
        return isolated_fallback_if_applicable()

    if choose_k(off_durations) != 2:
        return isolated_fallback_if_applicable()

    cluster_labels, cluster_centers = kmeans2(off_durations)
    short_gap_durations = off_durations[cluster_labels == 0]
    long_gap_durations  = off_durations[cluster_labels == 1]
    inter_element_mean  = float(np.mean(short_gap_durations))
    inter_burst_mean    = float(np.mean(long_gap_durations))

    if inter_element_mean <= 0:
        return isolated_fallback_if_applicable()

    burst_gap_ratio     = inter_burst_mean / inter_element_mean
    absolute_separation = inter_burst_mean - inter_element_mean
    long_gap_fraction   = len(long_gap_durations) / len(off_durations)
    n_long_gaps         = len(long_gap_durations)
    inter_burst_pixels  = inter_burst_mean / pixel_time

    if n_long_gaps == 1:
        single_long_gap_index = int(np.where(cluster_labels == 1)[0][0])
        single_gap_position   = single_long_gap_index / max(1, len(off_durations) - 1)
    else:
        single_gap_position = 0.5

    # See LONG_ELEMENT_GAP_SPECIES in the constants cell for the rationale.
    element_ratio_ok = (
        inter_burst_mean >= 2.5 * element_length
        and absolute_separation >= 1.5 * element_length
    ) or (
        relaxed_ratio and burst_gap_ratio >= 6.0 and absolute_separation >= 0.05
    )
    single_gap_min_pixels = 60 if relaxed_ratio else 200
    # ISOLATED_CALL_SPECIES clips are mostly single calls with a few real
    # doublets, so most gaps are "long" and the 0.55 ceiling wrongly rejects a
    # correct split. Relaxed here rather than routed to the fallback path.
    long_gap_fraction_ceiling = 0.90 if isolated_burst_ok else 0.55

    is_burst = (
        burst_gap_ratio     >= 4.0
        and element_ratio_ok
        and long_gap_fraction   <= long_gap_fraction_ceiling
        and inter_burst_pixels  >= 10
        and (n_long_gaps >= 2 or (inter_burst_pixels >= single_gap_min_pixels and single_gap_position >= 0.25))
    )

    if not is_burst:
        return isolated_fallback_if_applicable()

    burst_boundary_time = (inter_element_mean + inter_burst_mean) / 2

    bursts                = []
    current_on_durs       = []
    current_internal_gaps = []
    current_start_time    = 0.0
    elapsed               = 0.0

    for state, duration in time_bucket_list:
        if state == "On":
            if not current_on_durs:
                current_start_time = elapsed
            current_on_durs.append(duration)
        else:  # "Off"
            if duration > burst_boundary_time:
                if current_on_durs:
                    bursts.append({
                        'element_count':               len(current_on_durs),
                        'mean_element_length':          float(np.mean(current_on_durs)),
                        'mean_inter_element_interval':  (
                            float(np.mean(current_internal_gaps)) if current_internal_gaps else 0.0
                        ),
                        'start_time': current_start_time,
                        'end_time':   elapsed,
                    })
                current_on_durs, current_internal_gaps = [], []
            elif current_on_durs:
                current_internal_gaps.append(duration)
        elapsed += duration

    if current_on_durs:
        bursts.append({
            'element_count':               len(current_on_durs),
            'mean_element_length':          float(np.mean(current_on_durs)),
            'mean_inter_element_interval':  (
                float(np.mean(current_internal_gaps)) if current_internal_gaps else 0.0
            ),
            'start_time': current_start_time,
            'end_time':   elapsed,
        })

    if not bursts:
        return isolated_fallback_if_applicable()

    return inter_element_mean, inter_burst_mean, bursts


def classify_intervals(time_bucket_list, leading_silence, trailing_silence,
                       element_length, pixel_time, relaxed_ratio=False, isolated_burst_ok=False):
    """segment_bursts() collapsed to the original notebook's classify_intervals()
    return shape. Kept for parity; frog_process_multi uses segment_bursts +
    classify_burst_types directly. Returns (inter_element_interval,
    inter_burst_interval, elements_per_burst, min_elements_per_burst,
    max_elements_per_burst)."""
    inter_element_interval, inter_burst_interval, bursts = segment_bursts(
        time_bucket_list, leading_silence, trailing_silence, element_length, pixel_time,
        relaxed_ratio=relaxed_ratio, isolated_burst_ok=isolated_burst_ok,
    )
    counts = [b['element_count'] for b in bursts]
    median_epb = int(statistics.median(counts)) if counts else 1
    return (
        inter_element_interval, inter_burst_interval, max(1, median_epb),
        min(counts) if counts else 1, max(counts) if counts else 1,
    )

### Burst-Type Clustering

Takes the per-burst stats from `segment_bursts()` and groups the bursts into
types.

In [ ]:
def classify_burst_types(bursts, max_k=4, force_uniform=False, burst_frequencies=None):
    """
    Cluster this recording's bursts into distinct "types" -- the same k-means +
    silhouette technique as the gap classification above, generalized to
    multi-dimensional features and more than two clusters (kmeans_nd /
    silhouette_nd / choose_k_burst_types).

    Features per burst: element_count, mean_element_length,
    mean_inter_element_interval, and (if burst_frequencies is given) each
    burst's own dominant frequency in Hz -- z-score normalized. No
    amplitude/loudness -- by this stage the signal is already binary on/off,
    so none survives.

    Mutates and returns `bursts`, with each dict annotated with:
      'burst_type'    int, assigned in order of first appearance
      'burst_pattern' str, e.g. "ABCBA" -- same across every burst in the list

    With one burst, or no well-separated clusters (silhouette < 0.3), every
    burst gets type 0 ("A").

    force_uniform=True skips clustering and forces a single type -- for
    UNIFORM_BURST_TYPE_FILES, files confirmed to have only one real type where
    clustering was reading ordinary burst-to-burst jitter as a split. File-
    scoped rather than species-scoped, since the same species can show both a
    genuine split and a false one in different recordings.
    """
    n = len(bursts)
    if n < 2 or force_uniform:
        for b in bursts:
            b['burst_type']    = 0
            b['burst_pattern'] = 'A' * n
        return bursts

    feature_rows = [
        [b['element_count'], b['mean_element_length'], b['mean_inter_element_interval']]
        for b in bursts
    ]
    if burst_frequencies is not None:
        for row, freq in zip(feature_rows, burst_frequencies):
            row.append(freq)
    features = np.array(feature_rows, dtype=float)

    feature_std = features.std(axis=0)
    feature_std[feature_std == 0] = 1.0
    features_norm = (features - features.mean(axis=0)) / feature_std

    labels = choose_k_burst_types(features_norm, max_k=max_k)

    first_seen = {}
    relabeled  = []
    for lbl in labels:
        if lbl not in first_seen:
            first_seen[lbl] = len(first_seen)
        relabeled.append(first_seen[lbl])

    pattern = ''.join(chr(ord('A') + t) for t in relabeled)
    for b, t in zip(bursts, relabeled):
        b['burst_type']    = t
        b['burst_pattern'] = pattern

    return bursts

## Spectrogram Image Generation

Turns each cropped WAV file into an oscillogram PNG: find the dominant
frequency, bandpass filter around it to cut background noise, and save a
filled waveform with no axes.

In [ ]:
def find_dominant_frequency(signal, sample_rate):
    """Dominant FFT frequency of the clip above 500 Hz (the floor rejects mains
    hum and low-frequency environmental noise)."""
    spectrum   = np.abs(np.fft.rfft(signal))
    freqs      = np.fft.rfftfreq(len(signal), d=1 / sample_rate)
    above_500  = freqs > 500
    peak_idx   = np.argmax(spectrum[above_500])
    return freqs[above_500][peak_idx]


def compute_burst_frequencies(bursts, audio_signal, sample_rate, time_offset=0.0,
                               min_duration_s=0.02, min_range_hz=150.0):
    """Per-burst dominant frequency from the raw audio, as an optional 4th
    clustering feature in classify_burst_types--pitch distinguishes some
    burst types that count/length/spacing can't. Two guards against noise
    faking a distinction: a too-short burst falls back to the whole-clip
    frequency, and if all bursts land within min_range_hz the feature is
    dropped for this file (returns None)."""
    min_samples = max(1, round(min_duration_s * sample_rate))
    whole_clip_freq = find_dominant_frequency(audio_signal, sample_rate)
    freqs = []
    for b in bursts:
        start_sample = max(0, int((time_offset + b['start_time']) * sample_rate))
        end_sample   = min(len(audio_signal), int((time_offset + b['end_time']) * sample_rate))
        segment = audio_signal[start_sample:end_sample]
        freqs.append(find_dominant_frequency(segment, sample_rate) if len(segment) >= min_samples
                     else whole_clip_freq)
    if max(freqs) - min(freqs) < min_range_hz:
        return None
    return freqs


def generate_spectrogram_image(wav_path, output_path, figsize=(10, 2)):
    """
    Load a cropped WAV clip, bandpass-filter it around its dominant frequency,
    and save a filled oscillogram as a PNG image with no axes or padding.

    Parameters
    ----------
    wav_path    : str or Path  Path to the cropped WAV file in Cropped_Frogs/
    output_path : str or Path  Where to save the PNG image
    figsize     : tuple        Matplotlib figure size (width, height) in inches

    The saved image width in pixels = figsize[0] * SAVE_DPI.
    pixel_time in frog_process is then: clip_duration_s / image_width
    """
    signal, sample_rate = librosa.load(str(wav_path), sr=None)
    duration_seconds = len(signal) / sample_rate

    # ── Bandpass filter around dominant frequency ──────────────────────────────
    dominant_freq = find_dominant_frequency(signal, sample_rate)
    band_width_hz = 500  # Hz on each side of the dominant frequency

    # Use STFT to isolate the frequency band, then reconstruct via inverse STFT
    stft_matrix = librosa.stft(signal)
    magnitude   = np.abs(stft_matrix)
    phase       = np.angle(stft_matrix)
    frequencies = librosa.fft_frequencies(sr=sample_rate)

    freq_mask           = (
        (frequencies >= dominant_freq - band_width_hz) &
        (frequencies <= dominant_freq + band_width_hz)
    )
    magnitude_filtered  = magnitude * freq_mask[:, np.newaxis]

    # Suppress the quietest 80% of the filtered magnitude (remove residual noise)
    noise_floor = np.percentile(magnitude_filtered[magnitude_filtered > 0], 80)
    magnitude_filtered[magnitude_filtered < noise_floor] = 0

    filtered_signal = librosa.istft(magnitude_filtered * np.exp(1j * phase))

    # ── Draw and save oscillogram ──────────────────────────────────────────────
    time_axis = np.linspace(0, len(filtered_signal) / sample_rate, len(filtered_signal))

    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(time_axis, filtered_signal, color='black', linewidth=0.5)
    ax.fill_between(time_axis, filtered_signal, alpha=1.0, color='black')
    # Pin the x-axis to [0, duration]; otherwise matplotlib's autoscale margin
    # is baked into the PNG and every downstream pixel_time comes out ~9% short.
    ax.set_xlim(0, duration_seconds)
    ax.axis('off')
    plt.tight_layout(pad=0)

    fig.savefig(str(output_path), dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

## Missed-Element Rescue (file-specific)

Same issue as the katydid pipeline: image generation picks one dominant
frequency and noise floor for the whole recording, which can erase a real
secondary burst before the image is even drawn. The fix pulls the candidate
region straight from the raw audio and runs it through the normal detection
pipeline as its own crop, merging the result in afterward. The 48-file
whitelist below was built from 85 automated candidates, checked by hand.

One frog-specific wrinkle: frog oscillograms render at a fixed pixel width
rather than the katydid pipeline's duration-scaled one, so a rescue crop uses
its parent clip's own pixel density--otherwise a short crop would be way
over-sampled and lose genuinely short pulses.

Only runs when `frog_process_multi` is given a `wav_path` and the file is on
the whitelist; every other file is unaffected.

In [ ]:
# Files with a real secondary element/burst missing from the primary
# detection entirely -- confirmed by hand, one at a time, against waveform
# and amplitude data (see
MISSED_ELEMENT_RESCUE_FILES = {
    "Allobates_femoralis_55", "Alytes_obstetricans_53", "Arcovomer_passarellii_1",
    "Bufotes_viridis_62", "Dendropsophus_elegans_5", "Dendropsophus_elegans_7",
    "Dendropsophus_minutus_1", "Dendropsophus_rhodopeplus_1", "Elachistocleis_bicolor_4",
    "Elachistocleis_cesarii_1", "Elachistocleis_sikuani_1", "Fritziana_mitus_11",
    "Fritziana_mitus_12", "Haddadus_binotatus_3", "Hyla_arborea_86", "Hyla_arborea_90",
    "Hyla_molleri_14", "Hyla_molleri_16", "Hyla_molleri_23", "Hyloscirtus_alytolylax_1",
    "Leptobrachium_pullus_1", "Leptodactylus_fuscus_8", "Leptodactylus_podicipinus_1",
    "Leptodactylus_sertanejo_1", "Lithobates_berlandieri_1", "Litoria_burrowsi_1",
    "Microhyla_heymonsi_4", "Nyctixalus_pictus_1", "Pelophylax_esculentus_234",
    "Pelophylax_esculentus_98", "Pelophylax_lessonae_148", "Pelophylax_lessonae_24",
    "Pelophylax_lessonae_7", "Pelophylax_ridibundus_181", "Pelophylax_ridibundus_43",
    "Pelophylax_ridibundus_54", "Pithecopus_rohdei_5", "Polypedates_maculatus_2",
    "Pristimantis_labiosus_3", "Pristimantis_labiosus_4", "Pseudopaludicola_boliviana_1",
    "Rana_dalmatina_13", "Rana_dalmatina_21", "Rana_temporaria_66",
    "Rhinella_castaneotica_1", "Rhinella_ornata_4", "Scinax_fuscovarius_1",
    "Sphaenorhynchus_lacteus_9",
}

# Files confirmed to have only one real burst type -- classify_burst_types
# was reading ordinary burst-to-burst jitter as a real type split. See
UNIFORM_BURST_TYPE_FILES = {
    "Adenomera_marmorata_7", "Alytes_obstetricans_10",
    # Amerana_draytonii_282 deliberately left out even though it matches the
    # same pattern -- it has an older hand-fix that got lost in a CSV
    # regeneration, and force_uniform would paper over that with a different
    # wrong answer.; someone needs to redo the hand-fix.
    "Boana_pardalis_5", "Bufo_spinosus_12", "Bufo_spinosus_53",
    "Ololygon_argyreornata_3", "Pelodytes_ibericus_11", "Pelophylax_esculentus_3",
    "Pelophylax_perezi_14", "Rana_temporaria_96",
}


def render_raw_crop_png(signal_slice, sample_rate, output_path, target_pixel_time):
    """Render a rescue-candidate crop straight from the raw signal (not
    generate_spectrogram_image--its STFT gates are what may hide the missed
    element). Uses the parent clip's pixel_time so a short crop isn't
    over-sampled into losing its pulses."""
    duration_seconds = len(signal_slice) / sample_rate
    figure_width_inches = max(0.3, (duration_seconds / target_pixel_time) / SAVE_DPI)
    time_axis = np.linspace(0, duration_seconds, len(signal_slice))

    fig, ax = plt.subplots(figsize=(figure_width_inches, 2))
    ax.plot(time_axis, signal_slice, color='black', linewidth=0.5)
    ax.fill_between(time_axis, signal_slice, alpha=1.0, color='black')
    ax.set_xlim(0, duration_seconds)
    ax.axis('off')
    plt.tight_layout(pad=0)
    fig.savefig(str(output_path), dpi=SAVE_DPI, bbox_inches='tight', pad_inches=0)
    plt.close(fig)


def find_missed_element_regions(signal, sample_rate, primary_trim_sample_range,
                                 presence_multiplier=6.0, win_s=0.002,
                                 cluster_gap_s=0.015, isolation_gap_s=0.08,
                                 edge_buffer_s=0.25, pad_s=0.02,
                                 max_group_span_s=0.2,
                                 min_fraction_of_primary_peak=0.08,
                                 clip_edge_exclude_s=0.17):
    """Regions of raw-audio energy outside the primary detected region and
    separated from it by real quiet--candidate missed secondary bursts. Peak
    (not RMS) amplitude, since elements can be ~2 ms. for
    what each guard protects against."""
    win = max(1, int(win_s * sample_rate))
    n_win = len(signal) // win
    if n_win < 4:
        return []

    raw_env = np.array([
        np.max(np.abs(signal[j * win:(j + 1) * win])) for j in range(n_win)
    ])
    if raw_env.max() <= 0:
        return []

    baseline = np.percentile(raw_env, 10)
    if baseline <= 0:
        baseline = raw_env[raw_env > 0].min() if (raw_env > 0).any() else 1e-6

    ts0, ts1 = primary_trim_sample_range
    edge_buffer = int(edge_buffer_s * sample_rate)
    win0, win1 = max(0, (ts0 - edge_buffer) // win), min(n_win, (ts1 + edge_buffer) // win + 1)
    outside = np.ones(n_win, dtype=bool)
    outside[win0:win1] = False

    clip_edge_win = max(1, int(clip_edge_exclude_s / win_s))
    outside[:clip_edge_win] = False
    outside[max(0, n_win - clip_edge_win):] = False

    primary_peak = np.max(np.abs(signal[ts0:ts1])) if ts1 > ts0 else raw_env.max()
    min_absolute_peak = primary_peak * min_fraction_of_primary_peak

    real_energy = (raw_env > baseline * presence_multiplier) & (raw_env > min_absolute_peak)
    candidate = real_energy & outside
    if candidate.sum() < 1:
        return []

    gap_win = max(1, int(cluster_gap_s / win_s))
    idxs = np.where(candidate)[0]
    groups = []
    cur = [idxs[0]]
    for i in idxs[1:]:
        if i - cur[-1] <= gap_win:
            cur.append(i)
        else:
            groups.append(cur)
            cur = [i]
    groups.append(cur)

    confirmed = []
    quiet_thresh = baseline * 2.0
    min_quiet_run = max(1, int(isolation_gap_s / win_s))
    max_group_span_win = max(1, int(max_group_span_s / win_s))
    for group in groups:
        group_start, group_end = group[0], group[-1]
        if (group_end - group_start + 1) > max_group_span_win:
            continue
        ok = True
        if group_start >= win1:
            between = raw_env[win1:group_start] if group_start > win1 else np.array([])
        elif group_end < win0:
            between = raw_env[group_end + 1:win0] if win0 > group_end else np.array([])
        else:
            between = np.array([])
        if len(between):
            max_quiet_run = 0
            run = 0
            for level in between:
                if level < quiet_thresh:
                    run += 1
                    max_quiet_run = max(max_quiet_run, run)
                else:
                    run = 0
            if max_quiet_run < min_quiet_run:
                ok = False
        if ok:
            confirmed.append((group_start, group_end))

    pad = int(pad_s * sample_rate)
    regions = []
    for group_start, group_end in confirmed:
        region_start = max(0, group_start * win - pad)
        region_end = min(len(signal), (group_end + 1) * win + pad)
        regions.append((region_start, region_end))
    return regions


def analyze_missed_element_region(signal, sample_rate, region_start, region_end, target_pixel_time):
    """Render one confirmed rescue region as its own image and run the normal
    detection pipeline on it. Returns (time_buckets, region_start_seconds) in
    the clip's timeline, or None."""
    crop = signal[region_start:region_end]
    if len(crop) < 256:
        return None

    tmp_png_path = SPEC_DIR / '_missed_element_rescue_tmp.png'
    try:
        render_raw_crop_png(crop, sample_rate, tmp_png_path, target_pixel_time)
        spec_array = np.array(Image.open(str(tmp_png_path)).convert('L'))
        duration_seconds = len(crop) / sample_rate
        pixel_time = duration_seconds / spec_array.shape[1]
        ink = extract_outer_band_ink(spec_array)
        local_signal_list, _ = detect_signal_list_adaptive(ink, pixel_time)
    except (ValueError, ZeroDivisionError):
        return None
    finally:
        if tmp_png_path.exists():
            tmp_png_path.unlink()

    signal_columns = np.where(local_signal_list == 1)[0]
    if not len(signal_columns):
        return None
    local_trim_start = max(0, signal_columns[0] - 2)
    local_trim_end = min(len(local_signal_list), signal_columns[-1] + 3)
    cleaned = clean_signal_runs(
        local_signal_list[local_trim_start:local_trim_end], pixel_time, fill_gap_size=0)
    if not len(cleaned) or cleaned.mean() == 0:
        return None

    leading_offset_s = local_trim_start * pixel_time
    time_buckets = [
        ("On" if value == 1 else "Off", run_length * pixel_time)
        for value, run_length in rle(cleaned)
    ]
    region_start_s = region_start / sample_rate + leading_offset_s
    return time_buckets, region_start_s


def rescued_regions_to_bursts(regions_with_buckets):
    """Combine every confirmed rescue region's elements into one timeline and
    run segment_bursts once across all of them--not per region (which would
    split one coherent phrase into spurious 1-element bursts) and kept separate
    from the primary region's bursts. regions_with_buckets: (time_buckets,
    region_start_s) per region."""
    on_spans = []
    for time_buckets, region_start_s in regions_with_buckets:
        elapsed = 0.0
        for state, dur in time_buckets:
            if state == "On":
                on_spans.append((region_start_s + elapsed, region_start_s + elapsed + dur))
            elapsed += dur
    if not on_spans:
        return []
    on_spans.sort()

    combined_time_buckets = []
    prev_end = None
    for start, end in on_spans:
        if prev_end is not None:
            combined_time_buckets.append(("Off", start - prev_end))
        combined_time_buckets.append(("On", end - start))
        prev_end = end

    on_durations = [d for s, d in combined_time_buckets if s == "On"]
    element_length = float(np.mean(on_durations))
    _, _, bursts = segment_bursts(
        combined_time_buckets, 0.0, 0.0, element_length, 0.002,
        relaxed_ratio=False, isolated_burst_ok=False,
    )
    base_start = on_spans[0][0]
    for b in bursts:
        b['start_time'] += base_start
        b['end_time'] += base_start
    return bursts


# Files where two close elements were merged into one run by the primary
# threshold; split_signal_on_valleys splits them. File-scoped--as a default it
# fragmented ~80% of the dataset.
LOCAL_VALLEY_SPLIT_FILES = {
    "Acris_gryllus_1", "Acris_gryllus_6", "Adenomera_andreae_10", "Adenomera_heyeri_7",
    "Allobates_granti_7", "Allobates_insperatus_1", "Amerana_draytonii_101",
    "Amerana_draytonii_111", "Amietia_delalandii_1", "Boana_boans_4",
    "Boana_lanciformis_4", "Boana_platanera_2", "Brachycephalus_rotenbergae_2",
    "Brachycephalus_sulfuratus_1", "Bufo_bufo_18", "Bufo_bufo_26", "Bufo_bufo_41",
    "Bufo_bufo_47", "Calyptocephalella_gayi_1", "Crinia_deserticola_3",
    "Dendropsophus_microcephalus_7", "Dendropsophus_microps_2",
    "Dendropsophus_sarayacuensis_2", "Dryophytes_femoralis_1", "Dryophytes_versicolor_6",
    "Epidalea_calamita_18", "Epidalea_calamita_19", "Epidalea_calamita_20",
    "Epidalea_calamita_37", "Epidalea_calamita_69", "Hylarana_guentheri_6",
    "Hylarana_humeralis_2", "Hylarana_spinulosa_1", "Kassina_senegalensis_4",
    "Litoria_verreauxii_1", "Litoria_verreauxii_2", "Lysapsus_laevis_1",
    "Microhyla_fissipes_9", "Microhyla_heymonsi_1", "Nyctimystes_infrafrenatus_4",
    "Pelodytes_ibericus_14", "Pelodytes_ibericus_20", "Pelodytes_ibericus_31",
    "Pelodytes_punctatus_27", "Pelodytes_punctatus_8", "Pelophylax_esculentus_144",
    "Pelophylax_esculentus_224", "Pelophylax_esculentus_239",
    "Pelophylax_esculentus_279", "Pelophylax_esculentus_284", "Pelophylax_esculentus_35",
    "Pelophylax_esculentus_88", "Pelophylax_esculentus_94", "Pelophylax_grafi_4",
    "Pelophylax_lessonae_10", "Pelophylax_lessonae_179", "Pelophylax_lessonae_38",
    "Pelophylax_perezi_113", "Pelophylax_perezi_122", "Pelophylax_perezi_17",
    "Pelophylax_ridibundus_104", "Pelophylax_ridibundus_46", "Pelophylax_ridibundus_52",
    "Pseudacris_collinsorum_5", "Pseudacris_crucifer_41", "Pseudacris_feriarum_10",
    "Pseudacris_regilla_1", "Rana_dalmatina_7", "Rana_temporaria_110",
    "Rana_temporaria_206", "Rana_temporaria_25", "Rana_temporaria_33",
    "Rana_temporaria_42", "Raorchestes_annandalii_2", "Rhinella_marina_3",
    "Sonus_naturalis_3",
}

# Files with a real between-call gap the normal gap-duration clustering
# misses, confirmed by signal level during the gap (near zero, most of its
# length).
TRUE_SILENCE_SPLIT_FILES = {
    "Abavorana_luctuosa_1", "Acris_blanchardi_1", "Adenomera_nana_1",
    "Allobates_granti_25", "Alytes_obstetricans_16", "Alytes_obstetricans_38",
    "Alytes_obstetricans_47", "Alytes_obstetricans_6", "Ameerega_hahneli_20",
    "Ameerega_picta_1", "Ameerega_simulans_1", "Amerana_draytonii_102",
    "Amerana_draytonii_105", "Amerana_draytonii_125", "Amerana_draytonii_126",
    "Amerana_draytonii_127", "Amerana_draytonii_132", "Amerana_draytonii_133",
    "Amerana_draytonii_163", "Amerana_draytonii_164", "Amerana_draytonii_171",
    "Amerana_draytonii_173", "Amerana_draytonii_176", "Amerana_draytonii_186",
    "Amerana_draytonii_207", "Amerana_draytonii_215", "Amerana_draytonii_370",
    "Amerana_draytonii_371", "Amerana_draytonii_374", "Amerana_draytonii_38",
    "Amerana_draytonii_393", "Amerana_draytonii_395", "Amerana_draytonii_47",
    "Amerana_draytonii_48", "Amerana_draytonii_55", "Amerana_draytonii_56",
    "Amerana_draytonii_57", "Amerana_draytonii_58", "Amerana_draytonii_59",
    "Amerana_draytonii_65", "Amerana_draytonii_67", "Amerana_draytonii_68",
    "Amerana_draytonii_69", "Amerana_draytonii_70", "Amerana_draytonii_72",
    "Amerana_draytonii_73", "Amerana_draytonii_74", "Amerana_draytonii_80",
    "Amerana_draytonii_90", "Amerana_draytonii_92", "Amerana_draytonii_96",
    "Amerana_draytonii_97", "Anomaloglossus_surinamensis_2",
    "Aplastodiscus_albosignatus_6", "Aplastodiscus_lutzorum_1", "Aquarana_catesbeiana_1",
    "Aquarana_septentrionalis_3", "Boana_albomarginata_7", "Boana_faber_10",
    "Boana_faber_9", "Boana_jimenezi_1", "Boana_rosenbergi_4", "Boana_rufitela_2",
    "Bombina_bombina_21", "Bombina_bombina_49", "Bombina_variegata_16",
    "Brachycephalus_hermogenesi_5", "Bufo_bufo_24", "Bufo_bufo_39", "Bufo_bufo_51",
    "Bufo_bufo_59", "Bufo_bufo_62", "Bufo_bufo_67", "Bufo_bufo_69", "Bufo_spinosus_14",
    "Bufo_spinosus_16", "Bufo_spinosus_18", "Bufo_spinosus_19", "Bufo_spinosus_24",
    "Bufo_spinosus_29", "Bufo_spinosus_31", "Bufo_spinosus_43", "Bufo_spinosus_45",
    "Bufo_spinosus_48", "Bufo_spinosus_6", "Dendropsophus_columbianus_1",
    "Dendropsophus_mathiassoni_2", "Dendropsophus_walfordi_1",
    "Dendropsophus_walfordi_2", "Dendropsophus_walfordi_4", "Dryophytes_cinereus_1",
    "Dryophytes_cinereus_6", "Engystomops_pustulosus_1", "Gastrophryne_carolinensis_1",
    "Hyalinobatrachium_esmeralda_1", "Hyalinobatrachium_kawense_2", "Hyla_arborea_1",
    "Hyla_arborea_58", "Hyla_arborea_89", "Hyla_meridionalis_105", "Hyla_molleri_9",
    "Hylarana_guentheri_2", "Hylodes_phyllodes_5", "Hyloxalus_arliensis_1",
    "Hyloxalus_elachyhistus_1", "Hyloxalus_pulchellus_1", "Hyloxalus_sanctamariensis_1",
    "Hyperolius_platyceps_1", "Ischnocnema_lactea_1", "Ischnocnema_randorum_1",
    "Kalophrynus_baluensis_1", "Kaloula_pulchra_6", "Leptobrachium_ailaonicum_3",
    "Leptobrachium_bompu_2", "Leptobrachium_hainanense_1", "Leptodactylus_fragilis_2",
    "Leptodactylus_gracilis_1", "Leptodactylus_gracilis_2", "Leptodactylus_gracilis_4",
    "Leptodactylus_gracilis_5", "Leptodactylus_gracilis_6", "Leptodactylus_latinasus_1",
    "Leptodactylus_latinasus_2", "Leptodactylus_latinasus_3",
    "Leptodactylus_latinasus_5", "Leptodactylus_longirostris_1",
    "Leptodactylus_melanonotus_1", "Leptodactylus_melanonotus_2",
    "Leptodactylus_melanonotus_4", "Leptodactylus_validus_3", "Leptodactylus_wagneri_9",
    "Limnodynastes_convexiusculus_1", "Limnodynastes_dumerilii_4",
    "Lithobates_sphenocephalus_6", "Litoria_latopalmata_2", "Mixophyes_iteratus_2",
    "Myersiella_microps_1", "Ololygon_perpusilla_3", "Osteocephalus_oophagus_21",
    "Osteocephalus_taurinus_4", "Pelophylax_cypriensis_6", "Pelophylax_cypriensis_8",
    "Pelophylax_esculentus_134", "Pelophylax_esculentus_155",
    "Pelophylax_esculentus_174", "Pelophylax_esculentus_198",
    "Pelophylax_esculentus_202", "Pelophylax_esculentus_298",
    "Pelophylax_kurtmuelleri_11", "Pelophylax_kurtmuelleri_13",
    "Pelophylax_lessonae_202", "Pelophylax_lessonae_21", "Pelophylax_lessonae_8",
    "Pelophylax_lessonae_89", "Pelophylax_perezi_118", "Pelophylax_perezi_139",
    "Pelophylax_perezi_146", "Pelophylax_perezi_32", "Pelophylax_perezi_33",
    "Pelophylax_perezi_89", "Pelophylax_perezi_91", "Pelophylax_ridibundus_129",
    "Pelophylax_ridibundus_153", "Pelophylax_ridibundus_193",
    "Pelophylax_ridibundus_204", "Pelophylax_ridibundus_39", "Phasmahyla_guttata_4",
    "Philautus_tectus_1", "Physalaemus_nanus_2", "Platyplectrum_ornatum_2",
    "Polypedates_braueri_2", "Polypedates_leucomystax_2", "Pristimantis_frater_1",
    "Pristimantis_luteolateralis_1", "Pristimantis_myersi_1",
    "Pristimantis_unistrigatus_3", "Pseudacris_regilla_14", "Pseudacris_regilla_22",
    "Pseudophilautus_popularis_1", "Rana_arvalis_2", "Rana_arvalis_7",
    "Rana_dalmatina_14", "Raorchestes_gryllus_1", "Rhacophorus_edentulus_2",
    "Rhacophorus_reinwardtii_1", "Scinax_fuscovarius_2", "Scinax_nebulosus_4",
    "Scinax_x-signatus_1", "Smilisca_baudinii_2", "Smilisca_baudinii_3",
    "Tomopterna_cryptotis_1", "Vitreorana_parvula_1", "Wijayarana_masonii_4",
}


def true_silence_split(signal_list, normalized_signal_slice, pixel_time, element_length,
                        true_silence_level=0.05, sustained_level=0.15, sustained_fraction=0.70,
                        min_ratio=3.0):
    """
    Split signal_list at any gap whose signal both dips to true near-zero
    and stays quiet for most of its length (see TRUE_SILENCE_SPLIT_FILES
    above). Returns a list of (start_col, end_col) tuples, or None if no
    gap qualifies.
    """
    col = 0
    boundaries = []
    for value, run_length in rle(signal_list):
        if value == 0:
            seg = normalized_signal_slice[col:col + run_length]
            gap_duration = run_length * pixel_time
            if gap_duration >= min_ratio * element_length:
                sustained_frac = float((seg <= sustained_level).mean())
                if seg.min() <= true_silence_level and sustained_frac >= sustained_fraction:
                    boundaries.append((col, col + run_length))
        col += run_length

    if not boundaries:
        return None
    bursts = []
    prev = 0
    for (bstart, bend) in boundaries:
        bursts.append((prev, bstart))
        prev = bend
    bursts.append((prev, len(signal_list)))
    bursts = [(s, e) for s, e in bursts if e > s]
    return bursts if len(bursts) > 1 else None



def split_signal_on_valleys(raw_signal_list, normalized_signal, min_on_pixels,
                             prominence=0.18, min_side_peak=0.85):
    """
    Split an on-run at internal amplitude valleys deep enough to plausibly be
    a second close-together element merged into one run by the primary
    threshold. Only splits if both sides independently reach near-full
    amplitude, to avoid mistaking ordinary decay ripple for a second click.
    """
    out = raw_signal_list.copy()
    col = 0
    for value, run_length in rle(raw_signal_list):
        if value == 1 and run_length >= 2 * min_on_pixels:
            seg = normalized_signal[col:col + run_length]
            valleys, _ = find_peaks(-seg, prominence=prominence, distance=min_on_pixels)
            for v in valleys:
                if v < min_on_pixels or (run_length - v) < min_on_pixels:
                    continue
                left_peak = seg[:v].max()
                right_peak = seg[v:].max()
                if left_peak >= min_side_peak and right_peak >= min_side_peak:
                    out[col + v] = 0
        col += run_length
    return out


# Files confirmed to be continuous evenly-spaced croaking with no burst-level
# grouping; each element becomes its own burst (force_element_level_split).
# File-scoped--the same species shows both patterns.
ELEMENT_LEVEL_BURST_FILES = {
    "Acris_crepitans_2", "Acris_gryllus_4", "Acris_gryllus_7", "Adenomera_andreae_7",
    "Adenomera_heyeri_3", "Adenomera_heyeri_5", "Adenomera_heyeri_9",
    "Adenomera_hylaedactyla_1", "Adenomera_hylaedactyla_4", "Allobates_femoralis_73",
    "Allobates_femoralis_82", "Allobates_femoralis_83", "Allobates_zaparo_2",
    "Alytes_cisternasii_3", "Alytes_cisternasii_7", "Alytes_dickhilleni_10",
    "Ameerega_hahneli_12", "Ameerega_hahneli_13", "Ameerega_hahneli_18",
    "Ameerega_hahneli_21", "Ameerega_hahneli_23", "Ameerega_hahneli_9",
    "Ameerega_pulchripecta_1", "Amerana_draytonii_191", "Amerana_draytonii_210",
    "Amerana_draytonii_224", "Amerana_draytonii_236", "Amerana_draytonii_239",
    "Amerana_draytonii_260", "Amerana_draytonii_263", "Amerana_draytonii_268",
    "Amerana_draytonii_311", "Amerana_draytonii_312", "Amerana_draytonii_326",
    "Amerana_draytonii_328", "Amerana_draytonii_84", "Anaxyrus_americanus_11",
    "Anaxyrus_americanus_15", "Anaxyrus_americanus_16", "Anaxyrus_americanus_17",
    "Anaxyrus_cognatus_1", "Andinobates_supata_1", "Anomaloglossus_baeobatrachus_12",
    "Anomaloglossus_baeobatrachus_15", "Anomaloglossus_baeobatrachus_18",
    "Anomaloglossus_baeobatrachus_9", "Aplastodiscus_arildae_1",
    "Aplastodiscus_leucopygius_2", "Aquarana_clamitans_15", "Aquarana_clamitans_5",
    "Aquarana_clamitans_9", "Aquarana_grylio_11", "Aquarana_grylio_8",
    "Boana_calcarata_1", "Boana_calcarata_2", "Boana_faber_1", "Boana_faber_12",
    "Boana_pulchella_1", "Boana_rosenbergi_3", "Bokermannohyla_hylax_3",
    "Bombina_bombina_10", "Bombina_bombina_16", "Bombina_bombina_4",
    "Bombina_bombina_41", "Bombina_bombina_54", "Bombina_orientalis_1",
    "Bombina_variegata_1", "Bombina_variegata_2", "Bombina_variegata_23",
    "Bombina_variegata_4", "Boulenophrys_lushuiensis_1", "Bufo_bufo_11", "Bufo_bufo_12",
    "Bufo_bufo_32", "Bufo_gargarizans_1", "Bufo_spinosus_63", "Bufotes_pewzowi_2",
    "Bufotes_viridis_12", "Bufotes_viridis_13", "Bufotes_viridis_17",
    "Bufotes_viridis_2", "Bufotes_viridis_22", "Bufotes_viridis_30",
    "Bufotes_viridis_32", "Bufotes_viridis_36", "Bufotes_viridis_38",
    "Bufotes_viridis_41", "Bufotes_viridis_45", "Bufotes_viridis_50",
    "Bufotes_viridis_54", "Bufotes_viridis_61", "Bufotes_viridis_63",
    "Bufotes_viridis_65", "Bufotes_viridis_66", "Bufotes_viridis_7", "Bufotes_viridis_8",
    "Chaperina_fusca_1", "Chiasmocleis_haddadi_3", "Chiasmocleis_hudsoni_1",
    "Chiasmocleis_hudsoni_2", "Cochranella_granulosa_1", "Crinia_deserticola_1",
    "Crinia_signifera_2", "Ctenophryne_geayi_1", "Ctenophryne_geayi_2",
    "Cycloramphus_boraceiensis_3", "Dendropsophus_bogerti_1", "Dendropsophus_elegans_9",
    "Dendropsophus_gaucheri_1", "Dendropsophus_giesleri_4",
    "Dendropsophus_leucophyllatus_2", "Dendropsophus_microps_3",
    "Dendropsophus_minutus_11", "Dendropsophus_nanus_2", "Dryophytes_chrysoscelis_3",
    "Dryophytes_cinereus_2", "Dryophytes_gratiosus_1", "Dryophytes_japonicus_2",
    "Dryophytes_japonicus_3", "Duttaphrynus_melanostictus_10",
    "Duttaphrynus_melanostictus_4", "Duttaphrynus_melanostictus_5",
    "Duttaphrynus_melanostictus_6", "Duttaphrynus_scaber_1", "Duttaphrynus_scaber_2",
    "Duttaphrynus_scaber_3", "Eleutherodactylus_coqui_4",
    "Eleutherodactylus_johnstonei_9", "Epidalea_calamita_12", "Epidalea_calamita_24",
    "Epidalea_calamita_25", "Epidalea_calamita_28", "Epidalea_calamita_3",
    "Epidalea_calamita_30", "Epidalea_calamita_42", "Epidalea_calamita_43",
    "Epidalea_calamita_44", "Epidalea_calamita_49", "Epidalea_calamita_51",
    "Epidalea_calamita_94", "Fejervarya_limnocharis_5", "Fejervarya_multistriata_2",
    "Fejervarya_multistriata_6", "Fejervarya_multistriata_7", "Fritziana_goeldii_1",
    "Fritziana_goeldii_2", "Fritziana_goeldii_3", "Fritziana_goeldii_4",
    "Fritziana_goeldii_6", "Frostius_pernambucensis_3", "Gracixalus_medogensis_1",
    "Hoplobatrachus_crassus_2", "Hoplobatrachus_tigerinus_1",
    "Hyalinobatrachium_aureoguttatum_1", "Hyalinobatrachium_colymbiphyllum_1",
    "Hyalinobatrachium_kawense_4", "Hyla_annectans_1", "Hyla_arborea_10",
    "Hyla_arborea_100", "Hyla_arborea_104", "Hyla_arborea_11", "Hyla_arborea_12",
    "Hyla_arborea_13", "Hyla_arborea_19", "Hyla_arborea_2", "Hyla_arborea_21",
    "Hyla_arborea_22", "Hyla_arborea_23", "Hyla_arborea_27", "Hyla_arborea_30",
    "Hyla_arborea_32", "Hyla_arborea_33", "Hyla_arborea_36", "Hyla_arborea_37",
    "Hyla_arborea_39", "Hyla_arborea_41", "Hyla_arborea_47", "Hyla_arborea_48",
    "Hyla_arborea_49", "Hyla_arborea_50", "Hyla_arborea_53", "Hyla_arborea_54",
    "Hyla_arborea_55", "Hyla_arborea_57", "Hyla_arborea_59", "Hyla_arborea_63",
    "Hyla_arborea_65", "Hyla_arborea_68", "Hyla_arborea_69", "Hyla_arborea_7",
    "Hyla_arborea_70", "Hyla_arborea_72", "Hyla_arborea_73", "Hyla_arborea_76",
    "Hyla_arborea_78", "Hyla_arborea_79", "Hyla_arborea_81", "Hyla_arborea_83",
    "Hyla_arborea_9", "Hyla_arborea_95", "Hyla_arborea_96", "Hyla_arborea_97",
    "Hyla_arborea_99", "Hyla_intermedia_10", "Hyla_intermedia_13", "Hyla_intermedia_14",
    "Hyla_intermedia_3", "Hyla_intermedia_4", "Hyla_intermedia_5", "Hyla_intermedia_7",
    "Hyla_intermedia_9", "Hyla_meridionalis_62", "Hyla_meridionalis_66",
    "Hyla_meridionalis_70", "Hyla_meridionalis_98", "Hyla_meridionalis_99",
    "Hyla_molleri_10", "Hyla_molleri_11", "Hyla_molleri_15", "Hyla_molleri_17",
    "Hyla_molleri_5", "Hyla_orientalis_11", "Hyla_orientalis_13", "Hyla_orientalis_3",
    "Hyla_orientalis_6", "Hyla_sarda_10", "Hyla_sarda_11", "Hyla_sarda_13",
    "Hyla_sarda_14", "Hyla_sarda_17", "Hyla_sarda_20", "Hyla_sarda_23",
    "Hyla_savignyi_10", "Hyla_savignyi_12", "Hyla_savignyi_14", "Hyla_savignyi_15",
    "Hylarana_baramica_1", "Hylarana_daemeli_3", "Hylarana_guentheri_3",
    "Hylarana_guentheri_7", "Hylarana_nicobariensis_6", "Hylarana_sundabarat_1",
    "Hylodes_phyllodes_7", "Hyloscirtus_lindae_1", "Hyperolius_concolor_2",
    "Ingerophrynus_quadriporcatus_2", "Ischnocnema_guentheri_4",
    "Ischnocnema_guentheri_6", "Ischnocnema_henselii_1", "Ischnocnema_henselii_4",
    "Ischnocnema_henselii_5", "Ischnocnema_spanios_1", "Itapotihyla_langsdorffii_4",
    "Julianus_uruguayus_1", "Latonia_nigriventer_1", "Leptobrachium_abbotti_1",
    "Leptobrachium_hasseltii_4", "Leptodactylus_albilabris_2",
    "Leptodactylus_colombiensis_2", "Leptodactylus_fuscus_2",
    "Leptodactylus_intermedius_1", "Leptodactylus_knudseni_1",
    "Leptodactylus_mystaceus_1", "Leptodactylus_mystaceus_7",
    "Leptodactylus_rhodomystax_2", "Leptodactylus_rhodomystax_4",
    "Leptodactylus_rhodomystax_7", "Leptodactylus_savagei_1",
    "Leptodactylus_stenodema_2", "Leptodactylus_stenodema_4", "Leptodactylus_validus_2",
    "Leptodactylus_wagneri_2", "Leptomantis_cyanopunctatus_1", "Limnodynastes_peronii_4",
    "Limnonectes_microdiscus_2", "Limnonectes_microdiscus_4",
    "Limnonectes_microdiscus_5", "Lithobates_berlandieri_2", "Lithobates_palustris_1",
    "Lithobates_palustris_14", "Lithobates_palustris_6", "Lithobates_vaillanti_1",
    "Litoria_inermis_2", "Litoria_latopalmata_1", "Litoria_latopalmata_3",
    "Mannophryne_trinitatis_1", "Melanophryniscus_xanthostomus_1",
    "Metaphrynella_sundana_1", "Microhyla_heymonsi_11", "Microhyla_mukhlesuri_3",
    "Microhyla_ornata_6", "Minervarya_agricola_5", "Nyctimantis_rugiceps_2",
    "Nyctimantis_rugiceps_8", "Occidozyga_obscura_2", "Odontophrynus_americanus_1",
    "Ololygon_obtriangulata_2", "Oophaga_pumilio_1", "Oophaga_pumilio_2",
    "Oophaga_pumilio_5", "Oophaga_sylvatica_1", "Oophaga_sylvatica_2",
    "Oreophryne_variabilis_2", "Osteocephalus_leprieurii_1", "Osteocephalus_yasuni_1",
    "Pelobates_fuscus_11", "Pelobates_fuscus_7", "Pelobates_fuscus_9",
    "Pelodytes_ibericus_21", "Pelodytes_punctatus_26", "Pelodytes_punctatus_29",
    "Pelophylax_cypriensis_1", "Pelophylax_cypriensis_10", "Pelophylax_cypriensis_4",
    "Pelophylax_epeiroticus_3", "Pelophylax_esculentus_140", "Pelophylax_esculentus_216",
    "Pelophylax_esculentus_232", "Pelophylax_esculentus_240",
    "Pelophylax_esculentus_259", "Pelophylax_esculentus_263",
    "Pelophylax_esculentus_267", "Pelophylax_esculentus_268",
    "Pelophylax_esculentus_277", "Pelophylax_esculentus_282", "Pelophylax_esculentus_80",
    "Pelophylax_esculentus_85", "Pelophylax_lessonae_125", "Pelophylax_lessonae_132",
    "Pelophylax_lessonae_156", "Pelophylax_lessonae_223", "Pelophylax_lessonae_232",
    "Pelophylax_lessonae_236", "Pelophylax_lessonae_30", "Pelophylax_lessonae_35",
    "Pelophylax_lessonae_36", "Pelophylax_lessonae_51", "Pelophylax_lessonae_53",
    "Pelophylax_nigromaculatus_5", "Pelophylax_perezi_24", "Pelophylax_perezi_46",
    "Pelophylax_perezi_48", "Pelophylax_ridibundus_121", "Pelophylax_ridibundus_131",
    "Pelophylax_ridibundus_132", "Pelophylax_ridibundus_143",
    "Pelophylax_ridibundus_144", "Pelophylax_ridibundus_179",
    "Pelophylax_ridibundus_206", "Pelophylax_ridibundus_33", "Pelophylax_ridibundus_35",
    "Pelophylax_ridibundus_57", "Philautus_aurifasciatus_1", "Philautus_aurifasciatus_2",
    "Philautus_aurifasciatus_3", "Phrynobatrachus_bullans_1",
    "Phyllomedusa_burmeisteri_3", "Phyllomedusa_burmeisteri_4",
    "Physalaemus_atlanticus_2", "Physalaemus_fischeri_3", "Physalaemus_signifer_4",
    "Pithecopus_hypochondrialis_1", "Pleurodema_brachyops_3",
    "Polypedates_leucomystax_3", "Polypedates_leucomystax_6", "Polypedates_otilophus_1",
    "Pristimantis_chiastonotus_2", "Pristimantis_crepitaculus_1",
    "Pristimantis_espedeus_10", "Pristimantis_espedeus_17", "Pristimantis_espedeus_18",
    "Pristimantis_espedeus_19", "Pristimantis_espedeus_20", "Pristimantis_espedeus_23",
    "Pristimantis_espedeus_24", "Pristimantis_espedeus_29", "Pristimantis_espedeus_30",
    "Pristimantis_espedeus_31", "Pristimantis_espedeus_37", "Pristimantis_espedeus_38",
    "Pristimantis_espedeus_39", "Pristimantis_espedeus_41",
    "Pristimantis_luteolateralis_2", "Pristimantis_lymani_1", "Pristimantis_lymani_2",
    "Pristimantis_lymani_3", "Pristimantis_marmoratus_1", "Pristimantis_nyctophylax_1",
    "Pristimantis_paisa_1", "Pristimantis_unistrigatus_2", "Proceratophrys_boiei_1",
    "Pseudacris_clarkii_1", "Pseudacris_collinsorum_4", "Pseudacris_crucifer_1",
    "Pseudacris_crucifer_17", "Pseudacris_crucifer_3", "Pseudacris_crucifer_38",
    "Pseudacris_feriarum_1", "Pseudacris_feriarum_6", "Pseudacris_feriarum_9",
    "Pseudacris_maculata_13", "Pseudacris_maculata_9", "Pseudacris_ocularis_3",
    "Pseudacris_streckeri_1", "Pseudopaludicola_boliviana_2",
    "Pseudopaludicola_falcipes_1", "Pseudopaludicola_mystacalis_1",
    "Pseudopaludicola_saltica_1", "Pseudophilautus_reticulatus_1",
    "Pseudophryne_bibronii_2", "Rana_arvalis_10", "Rana_arvalis_13", "Rana_arvalis_15",
    "Rana_arvalis_16", "Rana_arvalis_17", "Rana_arvalis_30", "Rana_arvalis_35",
    "Rana_arvalis_36", "Rana_arvalis_38", "Rana_arvalis_39", "Rana_arvalis_47",
    "Rana_arvalis_5", "Rana_arvalis_55", "Rana_arvalis_60", "Rana_arvalis_62",
    "Rana_arvalis_64", "Rana_arvalis_9", "Rana_dalmatina_1", "Rana_dalmatina_12",
    "Rana_dalmatina_15", "Rana_dalmatina_16", "Rana_dalmatina_19", "Rana_dalmatina_27",
    "Rana_dalmatina_28", "Rana_dalmatina_29", "Rana_dalmatina_3", "Rana_dalmatina_31",
    "Rana_dalmatina_33", "Rana_dalmatina_34", "Rana_dalmatina_35", "Rana_dalmatina_39",
    "Rana_dalmatina_4", "Rana_dalmatina_40", "Rana_dalmatina_43", "Rana_dalmatina_44",
    "Rana_dalmatina_48", "Rana_dalmatina_49", "Rana_dalmatina_5", "Rana_dalmatina_6",
    "Rana_macrocnemis_1", "Rana_temporaria_118", "Rana_temporaria_132",
    "Rana_temporaria_133", "Rana_temporaria_142", "Rana_temporaria_144",
    "Rana_temporaria_149", "Rana_temporaria_152", "Rana_temporaria_165",
    "Rana_temporaria_166", "Rana_temporaria_17", "Rana_temporaria_172",
    "Rana_temporaria_173", "Rana_temporaria_177", "Rana_temporaria_180",
    "Rana_temporaria_181", "Rana_temporaria_183", "Rana_temporaria_184",
    "Rana_temporaria_185", "Rana_temporaria_186", "Rana_temporaria_188",
    "Rana_temporaria_189", "Rana_temporaria_19", "Rana_temporaria_192",
    "Rana_temporaria_2", "Rana_temporaria_203", "Rana_temporaria_208",
    "Rana_temporaria_21", "Rana_temporaria_22", "Rana_temporaria_23",
    "Rana_temporaria_36", "Rana_temporaria_51", "Rana_temporaria_54",
    "Rana_temporaria_61", "Rana_temporaria_76", "Rana_temporaria_81",
    "Rana_temporaria_83", "Ranoidea_gracilenta_1", "Ranoidea_longipes_1",
    "Ranoidea_maculosa_1", "Ranoidea_pearsoniana_1", "Raorchestes_sanctisilvaticus_2",
    "Rhacophorus_edentulus_4", "Rhacophorus_margaritifer_2", "Rhacophorus_rhodopus_1",
    "Rhaebo_guttatus_6", "Rhinella_arenarum_1", "Rhinella_arenarum_2",
    "Rhinella_arenarum_3", "Rhinella_crucifer_2", "Rhinella_diptycha_1",
    "Rhinella_horribilis_10", "Rhinella_horribilis_11", "Rhinella_horribilis_4",
    "Rhinella_horribilis_5", "Rhinella_horribilis_6", "Rhinella_icterica_2",
    "Rhinella_margaritifera_1", "Rhinella_marina_14", "Rhinella_marina_15",
    "Rhinella_marina_16", "Rhinella_marina_2", "Rhinella_marina_5", "Rhinella_marina_7",
    "Rhinella_marina_8", "Rhinella_ornata_6", "Scinax_boesemani_2", "Scinax_garbei_2",
    "Scinax_imbegue_1", "Scinax_imbegue_2", "Scinax_imbegue_7", "Scinax_nasicus_2",
    "Scinax_nebulosus_7", "Scinax_ruber_11", "Sclerophrys_gutturalis_1",
    "Sclerophrys_pusilla_1", "Silverstoneia_flotator_1", "Silverstoneia_flotator_2",
    "Silverstoneia_flotator_3", "Sonus_naturalis_21", "Sonus_naturalis_4",
    "Sphaenorhynchus_lacteus_2", "Sphaenorhynchus_lacteus_3", "Tepuihyla_tuberculosa_1",
    "Theloderma_licin_1", "Trachycephalus_resinifictrix_2",
    "Trachycephalus_resinifictrix_3", "Trachycephalus_typhonius_1", "Uperodon_systoma_1",
    "Vitreorana_ritae_3", "Wijayarana_masonii_1"
}


## Processing Function

`frog_process_multi` works like the original pipeline's `frog_process`, with
the low-volume rescue pass added in before silence-trimming. The difference:
it always returns a list of results, one per burst type, so callers can
just `rows.extend(...)` regardless of how many types a clip has.

In [ ]:
def try_split_on_true_silence(signal_list, normalized_signal, trim_start, trim_end,
                               pixel_time, element_length, min_ratio=3.0, sustained_fraction=0.70):
    """
    Try to split one detected burst into several at a real silence gap (see
    true_silence_split above). Returns (new_bursts, new_inter_burst_interval),
    or None if no gap qualified.
    """
    norm_slice = normalized_signal[trim_start:trim_end]
    split_ranges = true_silence_split(signal_list, norm_slice, pixel_time, element_length,
                                       min_ratio=min_ratio, sustained_fraction=sustained_fraction)
    if split_ranges is None:
        return None

    new_bursts = []
    for start_col, end_col in split_ranges:
        sub_signal = signal_list[start_col:end_col]
        sub_buckets = [('On' if value == 1 else 'Off', run_length * pixel_time)
                       for value, run_length in rle(sub_signal)]
        sub_on = [d for s, d in sub_buckets if s == 'On']
        if not sub_on:
            continue
        sub_off = [d for s, d in sub_buckets if s == 'Off']
        new_bursts.append({
            'element_count': len(sub_on),
            'mean_element_length': float(np.mean(sub_on)),
            'mean_inter_element_interval': float(np.mean(sub_off)) if sub_off else 0.0,
            'start_time': start_col * pixel_time,
            'end_time': end_col * pixel_time,
        })

    if len(new_bursts) <= 1:
        return None

    new_inter_burst_interval = float(np.mean([
        new_bursts[i + 1]['start_time'] - new_bursts[i]['end_time']
        for i in range(len(new_bursts) - 1)
    ]))
    return new_bursts, new_inter_burst_interval


def frog_process_multi(spectrogram_path, pixel_time, relaxed_ratio=False, isolated_burst_ok=False, wav_path=None):
    """Multi-burst-type variant of frog_process: ink → on/off signal → rescue
    quiet bursts → trim → segment into bursts → cluster into types. pixel_time
    is passed in (no OCR). wav_path, if given, enables the file-scoped
    whitelists (MISSED_ELEMENT_RESCUE_FILES etc.) and the missed-element rescue.
    Returns one dict per detected burst type (always a list)."""
    img = Image.open(str(spectrogram_path)).convert('L')
    spec_array = np.array(img)

    if spec_array.shape[1] == 0:
        raise ValueError("Empty image")

    ink = extract_outer_band_ink(spec_array)
    raw_signal_list, normalized_signal = detect_signal_list_adaptive(ink, pixel_time)
    raw_signal_list = rescue_quiet_bursts(spec_array, ink, raw_signal_list, pixel_time)

    if wav_path is not None and Path(wav_path).stem in LOCAL_VALLEY_SPLIT_FILES:
        min_on_pixels = max(1, round(0.003 / pixel_time))
        raw_signal_list = split_signal_on_valleys(raw_signal_list, normalized_signal, min_on_pixels)

    signal_columns = np.where(raw_signal_list == 1)[0]
    if not len(signal_columns):
        raise ValueError("No signal columns detected")

    trim_start = max(0, signal_columns[0] - 2)
    trim_end = min(len(raw_signal_list), signal_columns[-1] + 3)

    leading_silence = trim_start * pixel_time
    trailing_silence = (len(raw_signal_list) - trim_end) * pixel_time

    signal_list = clean_signal_runs(
        raw_signal_list[trim_start:trim_end], pixel_time, fill_gap_size=0
    )
    if not len(signal_list) or signal_list.mean() == 0:
        raise ValueError("Empty signal after cleanup")

    time_buckets = [
        ('On' if value == 1 else 'Off', run_length * pixel_time)
        for value, run_length in rle(signal_list)
    ]

    on_durations = [duration for state, duration in time_buckets if state == 'On']
    if not on_durations:
        raise ValueError("No on-pulses detected")
    element_length = float(np.mean(on_durations))

    _, inter_burst_interval, bursts = segment_bursts(
        time_buckets, leading_silence, trailing_silence, element_length, pixel_time,
        relaxed_ratio=relaxed_ratio, isolated_burst_ok=isolated_burst_ok,
        force_element_level_split=(wav_path is not None and Path(wav_path).stem in ELEMENT_LEVEL_BURST_FILES),
    )

    if (wav_path is not None and Path(wav_path).stem in TRUE_SILENCE_SPLIT_FILES
            and len(bursts) == 1):
        split_result = try_split_on_true_silence(
            signal_list, normalized_signal, trim_start, trim_end, pixel_time, element_length)
        if split_result is not None:
            bursts, inter_burst_interval = split_result

    # General fallback for files the manual whitelists haven't reached yet:
    # same real-gap-vs-continuous decision as the whitelist mechanisms above,
    # applied automatically. Never touches a file already handled above, or a
    # protected hand-fixed file.
    general_fallback_threshold = 20
    file_stem = Path(wav_path).stem if wav_path is not None else None
    general_fallback_element_level = False
    if (len(bursts) == 1
            and bursts[0]['element_count'] > general_fallback_threshold
            and file_stem is not None
            and file_stem not in PROTECTED_HAND_FIXED_FILES
            and file_stem not in TRUE_SILENCE_SPLIT_FILES
            and file_stem not in ELEMENT_LEVEL_BURST_FILES
            and file_stem not in MISSED_ELEMENT_RESCUE_FILES):
        split_result = try_split_on_true_silence(
            signal_list, normalized_signal, trim_start, trim_end, pixel_time, element_length)
        if split_result is not None:
            bursts, inter_burst_interval = split_result
        else:
            # No qualifying gap -- genuinely continuous. Represent it the
            # same way ELEMENT_LEVEL_BURST_FILES does: one micro-burst per
            # element, inter_burst_interval reported as 0.0.
            off_durations_only = [d for s, d in time_buckets if s == 'Off']
            mean_gap = float(np.mean(off_durations_only)) if off_durations_only else 0.0
            elapsed = 0.0
            elem_bursts = []
            for state, dur in time_buckets:
                if state == 'On':
                    elem_bursts.append({
                        'element_count': 1,
                        'mean_element_length': dur,
                        'mean_inter_element_interval': mean_gap,
                        'start_time': elapsed,
                        'end_time': elapsed + dur,
                    })
                elapsed += dur
            bursts = elem_bursts
            inter_burst_interval = 0.0
            general_fallback_element_level = True

    # Missed-element rescue (whitelisted files only) -- see the markdown cell above.
    raw_signal, sample_rate = None, None
    if wav_path is not None and Path(wav_path).stem in MISSED_ELEMENT_RESCUE_FILES:
        raw_signal, sample_rate = librosa.load(str(wav_path), sr=None)
        primary_range = (
            int(trim_start * pixel_time * sample_rate),
            int(trim_end * pixel_time * sample_rate),
        )
        rescue_regions = find_missed_element_regions(raw_signal, sample_rate, primary_range)
        regions_with_buckets = []
        for region_start, region_end in rescue_regions:
            result = analyze_missed_element_region(raw_signal, sample_rate, region_start, region_end, pixel_time)
            if result is not None:
                regions_with_buckets.append(result)
        rescued_bursts = rescued_regions_to_bursts(regions_with_buckets)
        if rescued_bursts:
            bursts = bursts + rescued_bursts
            bursts.sort(key=lambda b: b['start_time'])

    # Per-burst dominant frequency as a 4th clustering feature -- see
    # compute_burst_frequencies' docstring for the reliability guards.
    burst_frequencies = None
    if (wav_path is not None
            and Path(wav_path).stem.rsplit('_', 1)[0] not in FREQUENCY_FEATURE_EXCLUDED_SPECIES):
        if raw_signal is None:
            raw_signal, sample_rate = librosa.load(str(wav_path), sr=None)
        burst_frequencies = compute_burst_frequencies(bursts, raw_signal, sample_rate, time_offset=leading_silence)

    bursts = classify_burst_types(
        bursts,
        force_uniform=(general_fallback_element_level or
                       (wav_path is not None and Path(wav_path).stem in
                        (UNIFORM_BURST_TYPE_FILES | ELEMENT_LEVEL_BURST_FILES))),
        burst_frequencies=burst_frequencies,
    )

    n_types = len(set(b['burst_type'] for b in bursts))
    pattern = bursts[0]['burst_pattern']

    rows = []
    for burst_type in sorted(set(b['burst_type'] for b in bursts)):
        type_bursts = [b for b in bursts if b['burst_type'] == burst_type]
        counts = [b['element_count'] for b in type_bursts]
        rows.append({
            'element_length': round(
                float(np.mean([b['mean_element_length'] for b in type_bursts])), 4
            ),
            'inter_element_interval': round(
                float(np.mean([b['mean_inter_element_interval'] for b in type_bursts])), 4
            ),
            'inter_burst_interval': round(inter_burst_interval, 4),
            'elements_per_burst': int(statistics.median(counts)),
            'min_elements_per_burst': min(counts),
            'max_elements_per_burst': max(counts),
            'burst_type': chr(ord('A') + burst_type),
            'n_bursts_this_type': len(type_bursts),
            'n_burst_types_detected': n_types,
            'burst_pattern': pattern,
        })
    return rows


## Batch Run

For each clip: generate its oscillogram if needed, work out `pixel_time`, run
`frog_process_multi`, and collect the results--one clip can now produce
several rows. Images are the same shared cache the original pipeline uses;
results go to `frog_results_multi_bursts.csv`, a separate file that never
touches the original pipeline's output.

In [ ]:
rows = []

for _, clip_row in clips_ready.iterrows():
    wav_path       = Path(clip_row['clipped_file'])
    clip_duration  = float(clip_row['clip_duration_s'])
    genus          = clip_row['genus']
    species        = clip_row['species']

    # Confirmed noise-floor-jitter recordings (see NOISE_FLOOR_EXCLUDED_FILES
    # definition above) -- no meaningful element counts to report, skip entirely.
    if wav_path.stem in NOISE_FLOOR_EXCLUDED_FILES:
        continue

    spec_path = SPEC_DIR / (wav_path.stem + '_spectrogram.png')

    try:
        if not spec_path.exists():
            generate_spectrogram_image(wav_path, spec_path)

        img_width  = Image.open(str(spec_path)).size[0]
        pixel_time = clip_duration / img_width

        relaxed_ratio     = f"{genus}_{species}" in LONG_ELEMENT_GAP_SPECIES
        isolated_burst_ok = f"{genus}_{species}" in ISOLATED_CALL_SPECIES
        results = frog_process_multi(
            spec_path, pixel_time, relaxed_ratio=relaxed_ratio, isolated_burst_ok=isolated_burst_ok, wav_path=wav_path,
        )
        status  = 'ok'
        error   = ''

    except Exception as e:
        results = [{}]
        status  = 'error'
        error   = str(e)

    for result in results:
        rows.append({
            'genus':   genus,
            'species': species,
            'file':    wav_path.name,
            'status':  status,
            'error':   error,
            **result,
        })

output_csv  = SPEC_DIR / 'frog_results_multi_bursts.csv'
csv_columns = [
    'genus', 'species', 'file', 'status', 'error',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
    'burst_type', 'n_bursts_this_type', 'n_burst_types_detected', 'burst_pattern',
]
with open(output_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames = csv_columns, extrasaction = 'ignore')
    writer.writeheader()
    writer.writerows(rows)

successful   = [r for r in rows if r['status'] == 'ok']
errors       = [r for r in rows if r['status'] == 'error']
n_clips      = len({(r['genus'], r['species'], r['file']) for r in rows})
n_multi_type = len({
    (r['genus'], r['species'], r['file']) for r in rows
    if r['status'] == 'ok' and r.get('n_burst_types_detected', 1) > 1
})
print(f"Processed {n_clips} clips -> {len(rows)} rows ({len(successful)} ok, {len(errors)} errors)")
print(f"{n_multi_type} clip(s) had more than one detected burst type")
for r in errors:
    print(f"  ERROR {r['genus']} {r['species']} / {r['file']}: {r['error']}")

## Results

In [ ]:
df    = pd.read_csv(output_csv)
df_ok = df[df['status'] == 'ok'].copy()
print(f"{len(df_ok)} rows (across {df_ok[['genus','species','file']].drop_duplicates().shape[0]} clips) processed successfully")
df_ok[[
    'genus', 'species', 'burst_type', 'burst_pattern', 'n_bursts_this_type', 'n_burst_types_detected',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]].head(20)

## Build frog_final_multi DataFrame

Merges the processing results with Xeno-Canto metadata, same as the original
pipeline--no Temperature column, and `Amerana_draytonii` left blank since its
metadata can't be reliably matched back to individual clips. A clip with more
than one burst type produces more than one row here, all sharing the same
Genus/Species/File_ID.

Stored as `frog_final_multi`, kept separate from `frog_final`.

In [ ]:
# Derive File_ID from the WAV filename stem (e.g. 'Rana_temporaria_1' -> '1')
df_ok = df_ok.copy()
df_ok['File_ID'] = df_ok['file'].apply(
    lambda f: re.search(r'_(\d+)\.wav$', f).group(1)
    if re.search(r'_(\d+)\.wav$', f) else os.path.splitext(f)[0]
)

frog_final_multi = df_ok[[
    'genus', 'species', 'File_ID',
    'burst_type', 'burst_pattern', 'n_bursts_this_type', 'n_burst_types_detected',
    'element_length', 'inter_element_interval',
    'inter_burst_interval', 'elements_per_burst',
    'min_elements_per_burst', 'max_elements_per_burst',
]].rename(columns={
    'genus':                    'Genus',
    'species':                  'Species',
    'burst_type':                'Burst_Type',
    'burst_pattern':              'Burst_Pattern',
    'n_bursts_this_type':         'N_Bursts_This_Type',
    'n_burst_types_detected':     'N_Burst_Types_Detected',
    'element_length':            'Element_Length',
    'inter_element_interval':    'Inter-Element_Interval',
    'inter_burst_interval':      'Inter-Burst_Interval',
    'elements_per_burst':        'Elements_Per_Burst',
    'min_elements_per_burst':    'Min_Elements_Per_Burst',
    'max_elements_per_burst':    'Max_Elements_Per_Burst',
})

# Xeno-Canto metadata (Country/Location/Lat/Lon/Call Type): a lookup keyed by
# (Genus, Species, audio_num). Amerana_draytonii is left blank--its on-disk
# file count doesn't match frog_df, so the audio_num join isn't trustworthy.
frog_metadata = pd.read_csv(AUDIO_DIR / 'frog_metadata.csv')
frog_final_multi['audio_num'] = frog_final_multi['File_ID'].astype(int)
frog_final_multi = frog_final_multi.merge(
    frog_metadata, on=['Genus', 'Species', 'audio_num'], how='left'
).drop(columns='audio_num')

print(
    f'{len(frog_final_multi)} rows | {frog_final_multi["Genus"].nunique()} genera | '
    f'{frog_final_multi["Species"].nunique()} species | '
    f'{(frog_final_multi["N_Burst_Types_Detected"] > 1).sum()} rows from a multi-burst-type clip'
)
frog_final_multi.head()

In [ ]:
# Store separately from the original pipeline's frog_final -- this is an
# exploratory alternate output, not consumed by Display_Function.ipynb.
%store frog_final_multi

In [ ]:
frog_final_multi.to_csv(Path.home() / "Discrete_Signals" / "frog_multi_burst_type.csv", index=False)